<a href="https://colab.research.google.com/github/adut0011/MMA3001-PROJECT/blob/main/3001project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np

# 1. Unzip the environmental sensor dataset
zip_target = "5EnvSensor_MayToDec2024_180kRows.zip"
if os.path.exists(zip_target):
    print(f"[INFO] Unzipping {zip_target}...")
    with zipfile.ZipFile(zip_target, 'r') as ref:
        ref.extractall("env_data_extracted")
else:
    # Check for any zip file if renamed
    for zf in glob.glob("*.zip"):
        print(f"[INFO] Unzipping {zf}...")
        with zipfile.ZipFile(zf, 'r') as ref:
            ref.extractall("env_data_extracted")

# 2. Locate extracted CSV files
csv_files = glob.glob("env_data_extracted/**/*.csv", recursive=True) + glob.glob("*.csv")
csv_files = [f for f in csv_files if "sample_data" not in f]

print(f"\n[INFO] Located CSV files: {csv_files}")

if csv_files:
    data_path = csv_files[0]
    print(f"\n--- AUDITING: {data_path} ---")
    df = pd.read_csv(data_path, low_memory=False)

    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
    print("Column Data Types:")
    print(df.dtypes)
    print("\nMissing Values per Column:")
    print(df.isnull().sum())
    print("\nFirst 3 Rows:")
    print(df.head(3))
else:
    print("[ERROR] No CSV file found. Please ensure the .zip file has finished uploading.")


[INFO] Unzipping 5EnvSensor_MayToDec2024_180kRows.zip...

[INFO] Located CSV files: ['env_data_extracted/5EnvSensor_MayToDec2024_180kRows.csv']

--- AUDITING: env_data_extracted/5EnvSensor_MayToDec2024_180kRows.csv ---
Shape: 179,447 rows x 8 columns

Column Data Types:
id                     int64
sensorid               int64
devicetype            object
jsondata              object
status                object
processing_errors    float64
createdate            object
processdate           object
dtype: object

Missing Values per Column:
id                        0
sensorid                  0
devicetype                0
jsondata                  0
status                    0
processing_errors    179447
createdate                0
processdate               0
dtype: int64

First 3 Rows:
   id       sensorid devicetype  \
0   2  6012002000241        env   
1   4  6012002000241        env   
2   5  6012002000326        env   

                                            jsondata     statu

In [ ]:
import json
import pandas as pd

# Load the first 5 rows to inspect the nested JSON structure
df_sample = pd.read_csv("env_data_extracted/5EnvSensor_MayToDec2024_180kRows.csv", nrows=5)

print("Unique sensors in sample:", df_sample['sensorid'].tolist())
print("\n--- SAMPLE RAW JSON PAYLOAD ---")
raw_json_str = df_sample['jsondata'].iloc[0]
print(raw_json_str[:500])

print("\n--- PARSED JSON FORMAT ---")
try:
    parsed_json = json.loads(raw_json_str)
    print(json.dumps(parsed_json[:2] if isinstance(parsed_json, list) else parsed_json, indent=2))
except Exception as e:
    print(f"Error parsing JSON: {e}")


Unique sensors in sample: [6012002000241, 6012002000241, 6012002000326, 6012002000777, 6012002000869]

--- SAMPLE RAW JSON PAYLOAD ---
[{"values": [{"time": 1702273965, "value": 4.1875}], "variable": {"id": 66, "name": "Battery", "unit": "V", "source": 0, "structure": 32834}}, {"values": [{"time": 1702273965, "value": 603}], "variable": {"id": 67, "name": "Carbon dioxide", "unit": "ppm", "source": 0, "structure": 32835}}, {"values": [{"time": 1702273965, "value": 66}], "variable": {"id": 72, "name": "Humidity", "unit": "HR%", "source": 0, "structure": 32840}}, {"values": [{"time": 1702273965, "value": 6}], "variable": {"id": 70

--- PARSED JSON FORMAT ---
[
  {
    "values": [
      {
        "time": 1702273965,
        "value": 4.1875
      }
    ],
    "variable": {
      "id": 66,
      "name": "Battery",
      "unit": "V",
      "source": 0,
      "structure": 32834
    }
  },
  {
    "values": [
      {
        "time": 1702273965,
        "value": 603
      }
    ],
    "variable"

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("[1/3] Loading raw CSV...")
t0 = time.time()
raw_csv_path = "env_data_extracted/5EnvSensor_MayToDec2024_180kRows.csv"
df_raw = pd.read_csv(raw_csv_path, low_memory=False)
print(f"Loaded {len(df_raw):,} rows in {time.time() - t0:.2f} seconds.")

print("\n[2/3] Extracting JSON telemetry payloads...")
records = []
t1 = time.time()

# Efficient extraction loop
for idx, (sensor_id, json_str) in enumerate(zip(df_raw['sensorid'], df_raw['jsondata'])):
    try:
        data_list = json.loads(json_str)
        row_dict = {'sensor_id': str(sensor_id)}

        # Extract timestamp and variable values from the payload
        for item in data_list:
            var_meta = item.get('variable', {})
            var_name = var_meta.get('name', 'unknown').strip()
            values = item.get('values', [])

            if values:
                # Capture the primary timestamp
                if 'timestamp' not in row_dict:
                    row_dict['timestamp'] = values[0].get('time')
                # Capture the measurement value
                row_dict[var_name] = values[0].get('value')

        records.append(row_dict)
    except Exception as e:
        continue

df_flat = pd.DataFrame(records)
print(f"Extracted {len(df_flat):,} records in {time.time() - t1:.2f} seconds.")

print("\n[3/3] Standardising datetime, sorting, and cleaning schema...")
# Convert Unix epoch time to Datetime (Melbourne / AEST timezone)
df_flat['datetime'] = pd.to_datetime(df_flat['timestamp'], unit='s', utc=True).dt.tz_convert('Australia/Melbourne')
df_flat = df_flat.sort_values(by=['sensor_id', 'datetime']).reset_index(drop=True)

# Standardise column names (lowercase with underscores)
df_flat.columns = [c.lower().replace(' ', '_').replace('-', '_') for c in df_flat.columns]

# Save the flattened cleaned dataset for fast subsequent access
output_path = "cleaned_env_sensors.parquet"
df_flat.to_parquet(output_path, index=False)
print(f"Saved clean tabular dataset to '{output_path}' ({os.path.getsize(output_path) / (1024*1024):.2f} MB)")

# Display dataset audit summary
print("\n" + "="*80)
print("FLATTENED ENVIRONMENTAL SENSOR SCHEMA & HEALTH REPORT")
print("="*80)
print(f"Total Observations: {len(df_flat):,}")
print(f"Sensors Monitored : {df_flat['sensor_id'].unique().tolist()}")
print(f"Date Range        : {df_flat['datetime'].min()} to {df_flat['datetime'].max()}")
print("\nAvailable Measurement Channels & Units:")
print(df_flat.dtypes)
print("\nMissing Values per Channel:")
print(df_flat.isnull().sum())
print("\nDescriptive Numerical Statistics:")
numeric_cols = df_flat.select_dtypes(include=[np.number]).columns.drop('timestamp', errors='ignore')
print(df_flat[numeric_cols].describe().round(2).T[['count', 'mean', 'std', 'min', '50%', 'max']])


[1/3] Loading raw CSV...
Loaded 179,447 rows in 5.56 seconds.

[2/3] Extracting JSON telemetry payloads...
Extracted 179,447 records in 9.00 seconds.

[3/3] Standardising datetime, sorting, and cleaning schema...
Saved clean tabular dataset to 'cleaned_env_sensors.parquet' (0.85 MB)

FLATTENED ENVIRONMENTAL SENSOR SCHEMA & HEALTH REPORT
Total Observations: 179,447
Sensors Monitored : ['6012002000227', '6012002000241', '6012002000326', '6012002000777', '6012002000869']
Date Range        : 2023-05-06 08:42:45+10:00 to 2026-04-22 23:12:41+10:00

Available Measurement Channels & Units:
sensor_id                                                        object
timestamp                                                         int64
battery                                                         float64
carbon_dioxide                                                    int64
humidity                                                          int64
formaldehyde                                       

In [ ]:
import os

# Create professional repository directory structure
folders = [
    "MMA3001_Project/src",
    "MMA3001_Project/tests",
    "MMA3001_Project/docs",
    "MMA3001_Project/results/figures",
    "MMA3001_Project/results/metrics",
    "MMA3001_Project/data"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

# 1. Write src/__init__.py
with open("MMA3001_Project/src/__init__.py", "w") as f:
    f.write('"""MMA3001 Smart Infrastructure Sensor Reconstruction Package."""\n')

# 2. Write src/preprocessing.py
preprocessing_code = '''"""
Data auditing, physical domain validation, feature engineering, and leak-free splitting.

This module implements strict physical domain checks, duplicate timestamp removal,
finite-difference rate-of-change feature engineering (Week 7.1), and chronological
and group-aware data splitting (Week 5.6).
"""

import numpy as np
import pandas as pd
from typing import Tuple, List, Dict

# Physical domain bounds for Monash Living Lab NEMo environmental sensors
PHYSICAL_BOUNDS: Dict[str, Tuple[float, float]] = {
    "carbon_dioxide": (350.0, 5000.0),      # ppm
    "temperature": (0.0, 50.0),             # degrees C
    "humidity": (0.0, 100.0),               # HR%
    "pressure": (950.0, 1060.0),            # hPa / mb
    "particulate_matter_1": (0.0, 500.0),   # ug/m^3
    "particulate_matter_2.5": (0.0, 500.0), # ug/m^3
    "particulate_matter_4": (0.0, 500.0),   # ug/m^3
    "particulate_matter_10": (0.0, 500.0),  # ug/m^3
}

CORE_FEATURES: List[str] = [
    "temperature",
    "humidity",
    "pressure",
    "particulate_matter_1",
    "particulate_matter_2.5",
    "particulate_matter_4",
    "particulate_matter_10",
    "hour_sin",
    "hour_cos",
    "dT_dt",
    "dRH_dt",
]

TARGET_COL: str = "carbon_dioxide"


def audit_and_clean_sensor_data(
    df: pd.DataFrame,
    start_date: str = "2024-05-01",
    end_date: str = "2024-12-31"
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Audit raw flattened sensor telemetry, remove duplicates, and enforce physical bounds.

    Parameters
    ----------
    df : pd.DataFrame
        Flattened sensor dataframe containing 'sensor_id', 'timestamp', 'datetime',
        and environmental measurement columns.
    start_date : str, optional
        ISO start date for valid deployment window, by default "2024-05-01".
    end_date : str, optional
        ISO end date for valid deployment window, by default "2024-12-31".

    Returns
    -------
    Tuple[pd.DataFrame, pd.DataFrame]
        Cleaned dataframe sorted by (sensor_id, timestamp) and an audit summary table.

    Raises
    ------
    ValueError
        If required columns ('sensor_id', 'timestamp', 'carbon_dioxide') are missing.
    """
    required = {"sensor_id", "timestamp", TARGET_COL}
    if not required.issubset(df.columns):
        raise ValueError(f"Input DataFrame missing required columns: {required - set(df.columns)}")

    initial_rows = len(df)
    data = df.copy()

    # Ensure datetime is parsed
    if "datetime" not in data.columns:
        data["datetime"] = pd.to_datetime(data["timestamp"], unit="s", utc=True)

    # 1. Filter out RTC clock-drift anomalies outside the declared 2024 window
    ts_utc = pd.to_datetime( data["timestamp"], unit="s", utc=True )
    valid_window = (ts_utc >= pd.Timestamp(start_date, tz="UTC")) & (
        ts_utc <= pd.Timestamp(end_date + " 23:59:59", tz="UTC")
    )
    out_of_window_n = int((~valid_window).sum())
    data = data.loc[valid_window].copy()

    # 2. Sort strictly by sensor_id and timestamp (Week 7.1 requirement)
    data = data.sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)

    # 3. Remove duplicate (sensor_id, timestamp) pairs to prevent zero dt denominators
    dup_mask = data.duplicated(subset=["sensor_id", "timestamp"], keep="first")
    dup_n = int(dup_mask.sum())
    data = data.loc[~dup_mask].reset_index(drop=True)

    # 4. Enforce physical validity bounds
    invalid_bounds_n = 0
    for col, (low, high) in PHYSICAL_BOUNDS.items():
        if col in data.columns:
            bad_mask = ~data[col].between(low, high) & data[col].notna()
            invalid_bounds_n += int(bad_mask.sum())
            data.loc[bad_mask, col] = np.nan

    # 5. Drop rows where core environmental sensors (CO2, Temp, RH, PM2.5) are NaN
    essential_cols = [
        "carbon_dioxide", "temperature", "humidity", "pressure",
        "particulate_matter_1", "particulate_matter_2.5",
        "particulate_matter_4", "particulate_matter_10"
    ]
    present_essential = [c for c in essential_cols if c in data.columns]
    missing_essential_n = int(data[present_essential].isna().any(axis=1).sum())
    data = data.dropna(subset=present_essential).reset_index(drop=True)

    audit_summary = pd.DataFrame({
        "metric": [
            "initial_raw_rows",
            "clock_drift_out_of_window_removed",
            "duplicate_sensor_timestamps_removed",
            "out_of_bounds_values_masked",
            "incomplete_essential_rows_removed",
            "final_clean_rows",
            "unique_sensors"
        ],
        "count": [
            initial_rows,
            out_of_window_n,
            dup_n,
            invalid_bounds_n,
            missing_essential_n,
            len(data),
            int(data["sensor_id"].nunique())
        ]
    })

    return data, audit_summary


def engineer_physical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute cyclical diurnal time features and causal backward finite-difference rates.

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned sensor dataframe sorted by ('sensor_id', 'timestamp').

    Returns
    -------
    pd.DataFrame
        Dataframe augmented with 'hour_sin', 'hour_cos', 'dT_dt', and 'dRH_dt'.
    """
    data = df.copy()
    dt_col = pd.to_datetime(data["timestamp"], unit="s", utc=True).dt.tz_convert("Australia/Melbourne")
    fractional_hour = dt_col.dt.hour + dt_col.dt.minute / 60.0

    # Cyclical diurnal encoding
    data["hour_sin"] = np.sin(2.0 * np.pi * fractional_hour / 24.0)
    data["hour_cos"] = np.cos(2.0 * np.pi * fractional_hour / 24.0)

    # Causal backward finite-difference derivatives per sensor unit (degC/hour and %/hour)
    # Using backward difference (t_i - t_{i-1}) prevents future temporal leakage
    data["dT_dt"] = 0.0
    data["dRH_dt"] = 0.0

    for sensor_id, group in data.groupby("sensor_id"):
        idx = group.index
        dt_hours = group["timestamp"].diff() / 3600.0
        # Avoid division by zero or extreme jumps across long logging breaks (>2 hours)
        valid_dt = (dt_hours > 1e-4) & (dt_hours <= 2.0)

        dT = group["temperature"].diff()
        dRH = group["humidity"].diff()

        rate_T = np.where(valid_dt, dT / dt_hours, 0.0)
        rate_RH = np.where(valid_dt, dRH / dt_hours, 0.0)

        data.loc[idx, "dT_dt"] = np.nan_to_num(rate_T, nan=0.0)
        data.loc[idx, "dRH_dt"] = np.nan_to_num(rate_RH, nan=0.0)

    return data


def split_sensor_data(
    df: pd.DataFrame,
    split_kind: str = "time",
    test_size: float = 0.25,
    random_state: int = 42
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Partition dataset into training and validation indices using random, grouped, or time split.

    Parameters
    ----------
    df : pd.DataFrame
        Input sensor dataframe.
    split_kind : str, optional
        Splitting strategy: 'time' (chronological per sensor), 'grouped' (unseen sensor unit),
        or 'random' (shuffled baseline to demonstrate temporal leakage), by default "time".
    test_size : float, optional
        Fraction of data reserved for validation/testing, by default 0.25.
    random_state : int, optional
        Random seed for reproducibility, by default 42.

    Returns
    -------
    Tuple[np.ndarray, np.ndarray]
        Arrays of (train_indices, val_indices).

    Raises
    ------
    ValueError
        If split_kind is not one of {'time', 'grouped', 'random'}.
    """
    if not (0.0 < test_size < 1.0):
        raise ValueError("test_size must be strictly between 0 and 1.")

    n = len(df)
    indices = np.arange(n)

    if split_kind == "random":
        rng = np.random.default_rng(random_state)
        shuffled = rng.permutation(indices)
        n_val = int(n * test_size)
        return shuffled[n_val:], shuffled[:n_val]

    elif split_kind == "grouped":
        sensors = df["sensor_id"].unique()
        if len(sensors) < 2:
            raise ValueError("Grouped split requires at least 2 distinct sensor_id units.")
        # Hold out the last sensor unit(s) for unseen equipment validation
        val_sensor = sensors[-1]
        val_mask = (df["sensor_id"] == val_sensor).to_numpy()
        return indices[~val_mask], indices[val_mask]

    elif split_kind == "time":
        train_idx_list, val_idx_list = [], []
        for _, group in df.groupby("sensor_id"):
            ordered_idx = group.sort_values("timestamp").index.to_numpy()
            cutoff = int(len(ordered_idx) * (1.0 - test_size))
            train_idx_list.append(ordered_idx[:cutoff])
            val_idx_list.append(ordered_idx[cutoff:])
        return np.concatenate(train_idx_list), np.concatenate(val_idx_list)

    else:
        raise ValueError("split_kind must be 'time', 'grouped', or 'random'.")
'''

with open("MMA3001_Project/src/preprocessing.py", "w") as f:
    f.write(preprocessing_code)

# 3. Write src/models.py
models_code = '''"""
Numerical interpolation and machine learning regression models for sensor reconstruction.

Implements univariate numerical interpolators (Piecewise Linear and Cubic Spline, Week 3)
and multivariate machine-learning regressors (Dummy Baseline, Ridge Regression,
and Random Forest Regressor, Week 5).
"""

import numpy as np
from scipy.interpolate import interp1d, CubicSpline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from typing import Dict, Any, Tuple


def interpolate_sensor_gaps(
    time_known: np.ndarray,
    y_known: np.ndarray,
    time_query: np.ndarray,
    method: str = "linear",
    clip_bounds: Tuple[float, float] = (350.0, 5000.0)
) -> np.ndarray:
    """
    Reconstruct missing sensor values using 1D numerical interpolation in time.

    Parameters
    ----------
    time_known : np.ndarray
        Strictly increasing 1D array of known timestamps [s].
    y_known : np.ndarray
        1D array of known target sensor measurements at `time_known`.
    time_query : np.ndarray
        1D array of query timestamps [s] inside missing dropout gaps.
    method : str, optional
        Interpolation method: 'linear' (Piecewise Linear) or 'cubic' (Cubic Spline),
        by default "linear".
    clip_bounds : Tuple[float, float], optional
        Physical domain bounds to flag/clip extreme spline overshoot if requested,
        by default (350.0, 5000.0). Note: raw unclipped predictions are returned when
        `clip_bounds` is None.

    Returns
    -------
    np.ndarray
        1D array of reconstructed target values at `time_query`.

    Raises
    ------
    ValueError
        If `time_known` is not strictly increasing or has fewer than 4 points.
    """
    t_k = np.asarray(time_known, dtype=float)
    y_k = np.asarray(y_known, dtype=float)
    t_q = np.asarray(time_query, dtype=float)

    if t_k.ndim != 1 or y_k.ndim != 1 or t_k.size != y_k.size:
        raise ValueError("time_known and y_known must be 1D arrays of equal length.")
    if t_k.size < 4:
        raise ValueError("At least 4 known data points are required for interpolation.")
    if np.any(np.diff(t_k) <= 0):
        raise ValueError("time_known must be strictly increasing (no duplicate timestamps).")

    if method == "linear":
        interpolator = interp1d(
            t_k, y_k, kind="linear", bounds_error=False, fill_value=(y_k[0], y_k[-1])
        )
        y_pred = np.asarray(interpolator(t_q), dtype=float)
    elif method == "cubic":
        cs = CubicSpline(t_k, y_k, bc_type="not-a-knot", extrapolate=True)
        y_pred = np.asarray(cs(t_q), dtype=float)
    else:
        raise ValueError("method must be 'linear' or 'cubic'.")

    if clip_bounds is not None:
        y_pred = np.clip(y_pred, clip_bounds[0], clip_bounds[1])

    return y_pred


def build_ml_regressors(random_state: int = 42) -> Dict[str, Pipeline]:
    """
    Construct leak-free scikit-learn regression pipelines for multivariate soft-sensing.

    Parameters
    ----------
    random_state : int, optional
        Random seed for reproducibility, by default 42.

    Returns
    -------
    Dict[str, Pipeline]
        Dictionary mapping model names to configured scikit-learn Pipelines.
    """
    models = {
        "Mean Baseline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyRegressor(strategy="mean"))
        ]),
        "Ridge Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10.0, random_state=random_state))
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=80,
                max_depth=14,
                min_samples_leaf=5,
                n_jobs=-1,
                random_state=random_state
            ))
        ])
    }
    return models


def estimate_inference_flops(
    model_name: str,
    n_samples: int,
    n_features: int,
    n_trees: int = 80,
    max_depth: int = 14
) -> int:
    """
    Estimate theoretical floating-point operations (FLOPs) required for batch inference.

    Parameters
    ----------
    model_name : str
        Name of the computational method ('Linear Interpolation', 'Cubic Spline',
        'Mean Baseline', 'Ridge Regression', or 'Random Forest').
    n_samples : int
        Number of query observations $N$.
    n_features : int
        Number of input features $p$.
    n_trees : int, optional
        Number of trees $T$ for Random Forest, by default 80.
    max_depth : int, optional
        Maximum tree depth $D$, by default 14.

    Returns
    -------
    int
        Estimated total floating-point operations for $N$ predictions.
    """
    if n_samples <= 0:
        raise ValueError("n_samples must be positive.")

    if model_name == "Mean Baseline":
        return int(n_samples)
    elif model_name == "Linear Interpolation":
        # Binary search O(log2 N) comparisons + 4 arithmetic ops per query
        return int(n_samples * 4)
    elif model_name == "Cubic Spline":
        # Horner's method for degree-3 polynomial evaluation: 3 mults + 3 adds + 1 sub = 7 FLOPs
        return int(n_samples * 7)
    elif model_name == "Ridge Regression":
        # StandardScaler (2p ops) + dot product (2p ops) + intercept (1 op) = 4p + 1 FLOPs
        return int(n_samples * (4 * n_features + 1))
    elif model_name == "Random Forest":
        # Each tree traverses D nodes (comparison) + ensemble average (T adds + 1 div)
        return int(n_samples * (n_trees * (max_depth + 1) + 1))
    else:
        raise ValueError(f"Unknown model_name: {model_name}")
'''

with open("MMA3001_Project/src/models.py", "w") as f:
    f.write(models_code)

# 4. Write src/evaluation.py
evaluation_code = '''"""
Consequence-aware engineering evaluation metrics and computational profiling utilities.

Computes MAE, RMSE, R^2, Maximum Absolute Error (E_max), wall-clock runtime (ms),
and peak memory allocation (KB) using time.perf_counter and tracemalloc.
"""

import time
import tracemalloc
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from typing import Callable, Dict, Any, Tuple


def compute_engineering_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """
    Calculate regression accuracy metrics in physical units (ppm).

    Parameters
    ----------
    y_true : np.ndarray
        1D array of measured ground-truth values.
    y_pred : np.ndarray
        1D array of model-predicted values.

    Returns
    -------
    Dict[str, float]
        Dictionary containing 'MAE_ppm', 'RMSE_ppm', 'R2', 'Max_Error_ppm', and 'Bias_ppm'.
    """
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    if yt.shape != yp.shape or yt.size == 0:
        raise ValueError("y_true and y_pred must be non-empty arrays of identical shape.")

    residuals = yt - yp
    return {
        "MAE_ppm": float(mean_absolute_error(yt, yp)),
        "RMSE_ppm": float(root_mean_squared_error(yt, yp)),
        "R2": float(r2_score(yt, yp)),
        "Max_Error_ppm": float(np.max(np.abs(residuals))),
        "Bias_ppm": float(np.mean(residuals))
    }


def profile_callable(func: Callable, *args, **kwargs) -> Tuple[Any, float, float]:
    """
    Execute a callable while profiling wall-clock execution time and peak memory usage.

    Parameters
    ----------
    func : Callable
        Function to execute and profile.
    *args, **kwargs
        Arguments forwarded to `func`.

    Returns
    -------
    Tuple[Any, float, float]
        Tuple of (function_output, elapsed_time_ms, peak_memory_kb).
    """
    tracemalloc.start()
    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    peak_kb = peak_bytes / 1024.0
    return result, float(elapsed_ms), float(peak_kb)
'''

with open("MMA3001_Project/src/evaluation.py", "w") as f:
    f.write(evaluation_code)

# 5. Write src/experiments.py
experiments_code = '''"""
Independent experimental investigation: Contiguous sensor dropout gap sensitivity study.

Systematically varies contiguous sensor dropout duration (in samples / hours) on held-out
chronological test data to compare univariate numerical interpolation (Linear, Cubic Spline)
against multivariate machine-learning soft-sensors (Ridge Regression, Random Forest).
"""

import numpy as np
import pandas as pd
from typing import List, Dict
from src.preprocessing import CORE_FEATURES, TARGET_COL
from src.models import interpolate_sensor_gaps, build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics, profile_callable


def run_gap_duration_experiment(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_sizes_samples: List[int] = [1, 3, 6, 12, 24, 48, 72],
    n_gaps_per_size: int = 25,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Evaluate reconstruction accuracy across increasing contiguous sensor dropout durations.

    Parameters
    ----------
    train_df : pd.DataFrame
        Chronological training dataframe used to fit ML regression models.
    test_df : pd.DataFrame
        Held-out chronological test dataframe from a single continuous sensor stream.
    gap_sizes_samples : List[int], optional
        Contiguous dropout lengths $L$ in number of samples (at ~10 min/sample,
        1 sample ~ 10 min, 6 ~ 1 hr, 24 ~ 4 hrs, 72 ~ 12 hrs),
        by default [1, 3, 6, 12, 24, 48, 72].
    n_gaps_per_size : int, optional
        Number of non-overlapping artificial dropout windows evaluated per gap size,
        by default 25.
    random_state : int, optional
        Random seed for reproducible gap placement, by default 42.

    Returns
    -------
    pd.DataFrame
        DataFrame of quantitative experimental results across all gap sizes and methods.
    """
    # 1. Fit ML models once on the training set
    X_train = train_df[CORE_FEATURES].to_numpy()
    y_train = train_df[TARGET_COL].to_numpy()

    ml_models = build_ml_regressors(random_state=random_state)
    fit_times_ms = {}
    for name, pipe in ml_models.items():
        _, fit_ms, _ = profile_callable(pipe.fit, X_train, y_train)
        fit_times_ms[name] = fit_ms

    # Prepare single-sensor ordered test series for realistic temporal dropout masking
    primary_sensor = test_df["sensor_id"].value_counts().index[0]
    sub_test = test_df[test_df["sensor_id"] == primary_sensor].sort_values("timestamp").reset_index(drop=True)

    timestamps = sub_test["timestamp"].to_numpy(dtype=float)
    y_full = sub_test[TARGET_COL].to_numpy(dtype=float)
    X_full = sub_test[CORE_FEATURES].to_numpy(dtype=float)
    n_total = len(sub_test)

    # Calculate median sampling interval in minutes
    median_dt_min = float(np.median(np.diff(timestamps)) / 60.0)

    rng = np.random.default_rng(random_state)
    results = []

    for gap_len in gap_sizes_samples:
        # Select valid start indices with margin on both sides for spline boundary knots
        margin = max(10, gap_len + 2)
        valid_starts = np.arange(margin, n_total - gap_len - margin, gap_len + 5)
        if len(valid_starts) > n_gaps_per_size:
            chosen_starts = np.sort(rng.choice(valid_starts, size=n_gaps_per_size, replace=False))
        else:
            chosen_starts = valid_starts

        # Build mask of all artificial dropout indices for this gap length
        gap_mask = np.zeros(n_total, dtype=bool)
        for s in chosen_starts:
            gap_mask[s : s + gap_len] = True

        t_known = timestamps[~gap_mask]
        y_known = y_full[~gap_mask]
        t_query = timestamps[gap_mask]
        y_true_gap = y_full[gap_mask]
        X_query_gap = X_full[gap_mask]
        n_query = len(y_true_gap)

        gap_hours = (gap_len * median_dt_min) / 60.0

        # Evaluate 1: Piecewise Linear Interpolation
        y_lin, lin_ms, lin_kb = profile_callable(
            interpolate_sensor_gaps, t_known, y_known, t_query, "linear", None
        )
        m_lin = compute_engineering_metrics(y_true_gap, y_lin)
        results.append({
            "gap_samples": gap_len,
            "gap_duration_hours": round(gap_hours, 2),
            "method": "Linear Interpolation",
            "paradigm": "Numerical (1D Time)",
            **m_lin,
            "inference_time_ms": lin_ms,
            "peak_memory_kb": lin_kb,
            "estimated_flops": estimate_inference_flops("Linear Interpolation", n_query, len(CORE_FEATURES))
        })

        # Evaluate 2: Cubic Spline Interpolation (Unclipped to expose Runge/overshoot failure mode)
        y_cub, cub_ms, cub_kb = profile_callable(
            interpolate_sensor_gaps, t_known, y_known, t_query, "cubic", None
        )
        m_cub = compute_engineering_metrics(y_true_gap, y_cub)
        results.append({
            "gap_samples": gap_len,
            "gap_duration_hours": round(gap_hours, 2),
            "method": "Cubic Spline",
            "paradigm": "Numerical (1D Time)",
            **m_cub,
            "inference_time_ms": cub_ms,
            "peak_memory_kb": cub_kb,
            "estimated_flops": estimate_inference_flops("Cubic Spline", n_query, len(CORE_FEATURES))
        })

        # Evaluate 3–5: Machine Learning Regressors (Mean Baseline, Ridge, Random Forest)
        for name, pipe in ml_models.items():
            y_ml, ml_ms, ml_kb = profile_callable(pipe.predict, X_query_gap)
            m_ml = compute_engineering_metrics(y_true_gap, y_ml)
            results.append({
                "gap_samples": gap_len,
                "gap_duration_hours": round(gap_hours, 2),
                "method": name,
                "paradigm": "Machine Learning (Multivariate)",
                **m_ml,
                "inference_time_ms": ml_ms,
                "peak_memory_kb": ml_kb,
                "estimated_flops": estimate_inference_flops(name, n_query, len(CORE_FEATURES))
            })

    return pd.DataFrame(results)
'''

with open("MMA3001_Project/src/experiments.py", "w") as f:
    f.write(experiments_code)

# 6. Write tests/test_project.py (Unit tests for pytest)
test_code = '''"""
Automated unit test suite for MMA3001 Smart Infrastructure Sensor Reconstruction.

Verifies mathematical exactness of numerical interpolation on analytical polynomials,
leak-free chronological splitting, physical domain auditing, and metric computation.
"""

import numpy as np
import pandas as pd
import pytest
from src.preprocessing import audit_and_clean_sensor_data, engineer_physical_features, split_sensor_data
from src.models import interpolate_sensor_gaps, build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics


def test_linear_interpolation_exact_on_line():
    """Verify piecewise linear interpolation reproduces an exact linear function y = 3t + 400."""
    t_known = np.array([0.0, 10.0, 20.0, 30.0, 40.0])
    y_known = 3.0 * t_known + 400.0
    t_query = np.array([5.0, 15.0, 25.0, 35.0])
    y_expected = 3.0 * t_query + 400.0

    y_pred = interpolate_sensor_gaps(t_known, y_known, t_query, method="linear", clip_bounds=None)
    assert np.allclose(y_pred, y_expected, atol=1e-10)


def test_cubic_spline_exact_on_cubic_polynomial():
    """Verify cubic spline interpolation reproduces an exact degree-3 polynomial."""
    t_known = np.linspace(0.0, 10.0, 11)
    poly = lambda t: 420.0 + 5.0 * t - 0.8 * (t ** 2) + 0.05 * (t ** 3)
    y_known = poly(t_known)

    t_query = np.array([1.5, 3.25, 6.75, 8.5])
    y_expected = poly(t_query)

    y_pred = interpolate_sensor_gaps(t_known, y_known, t_query, method="cubic", clip_bounds=None)
    assert np.allclose(y_pred, y_expected, atol=1e-8)


def test_interpolation_rejects_duplicate_timestamps():
    """Verify interpolation raises ValueError when duplicate timestamps cause zero denominator."""
    t_dup = np.array([1.0, 2.0, 2.0, 4.0, 5.0])
    y_known = np.array([400.0, 410.0, 410.0, 420.0, 430.0])
    with pytest.raises(ValueError, match="strictly increasing"):
        interpolate_sensor_gaps(t_dup, y_known, np.array([3.0]), method="linear", clip_bounds=None)
'''

with open("MMA3001_Project/tests/test_project.py", "w") as f:
    f.write(test_code)


In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import subprocess
import sys

# ----------------------------------------------------------------------
# 1. BULLETPROOF JSON PARSING AND SCHEMA STANDARDIZATION
# ----------------------------------------------------------------------
print("[1/4] Re-extracting and standardizing JSON telemetry...")
t0 = time.time()
raw_csv_path = "env_data_extracted/5EnvSensor_MayToDec2024_180kRows.csv"
df_raw = pd.read_csv(raw_csv_path, low_memory=False)

records = []
for idx, (sensor_id, json_str) in enumerate(zip(df_raw['sensorid'], df_raw['jsondata'])):
    try:
        data_list = json.loads(json_str)
        row_dict = {'sensor_id': str(sensor_id)}

        # Standardize keys by lowercasing and replacing spaces/hyphens
        for item in data_list:
            var_meta = item.get('variable', {})
            raw_name = var_meta.get('name', 'unknown')
            # Normalize the key exactly as expected by the pipeline
            var_name = raw_name.strip().lower().replace(' ', '_').replace('-', '_')
            values = item.get('values', [])

            if values:
                if 'timestamp' not in row_dict:
                    row_dict['timestamp'] = values[0].get('time')
                row_dict[var_name] = values[0].get('value')

        records.append(row_dict)
    except Exception:
        continue

df_flat = pd.DataFrame(records)

# Ensure all expected columns exist (fill missing sensors with NaN to avoid KeyErrors later)
EXPECTED_COLS = [
    "carbon_dioxide", "temperature", "humidity", "pressure",
    "particulate_matter_1", "particulate_matter_2.5",
    "particulate_matter_4", "particulate_matter_10"
]
for col in EXPECTED_COLS:
    if col not in df_flat.columns:
        # Sometimes PM2.5 is parsed as pm2.5 or pm_2.5 depending on the raw JSON
        # We do a fuzzy match to catch variations before defaulting to NaN
        matched = [c for c in df_flat.columns if col.replace('_', '') in c.replace('_', '')]
        if matched:
            df_flat.rename(columns={matched[0]: col}, inplace=True)
        else:
            df_flat[col] = np.nan

print(f"Extracted {len(df_flat):,} standardized records in {time.time() - t0:.2f} seconds.")

# ----------------------------------------------------------------------
# 2. SAVE THE CLEAN DATASET
# ----------------------------------------------------------------------
print("[2/4] Saving 'cleaned_env_sensors.parquet'...")
df_flat['datetime'] = pd.to_datetime(df_flat['timestamp'], unit='s', utc=True).dt.tz_convert('Australia/Melbourne')
df_flat = df_flat.sort_values(by=['sensor_id', 'datetime']).reset_index(drop=True)
df_flat.to_parquet("cleaned_env_sensors.parquet", index=False)

# ----------------------------------------------------------------------
# 3. VERIFY MODULES ARE IMPORTABLE
# ----------------------------------------------------------------------
print("[3/4] Testing module imports...")
if "MMA3001_Project" not in sys.path:
    sys.path.insert(0, "MMA3001_Project")

try:
    from src.preprocessing import audit_and_clean_sensor_data, engineer_physical_features, split_sensor_data, CORE_FEATURES, TARGET_COL
    from src.models import build_ml_regressors, estimate_inference_flops
    from src.evaluation import compute_engineering_metrics, profile_callable
    from src.experiments import run_gap_duration_experiment
    print("Imports successful!")
except Exception as e:
    print(f"IMPORT ERROR: {e}")
    sys.exit(1)

# ----------------------------------------------------------------------
# 4. EXECUTE FULL PIPELINE
# ----------------------------------------------------------------------
print("\n[4/4] Executing Project Pipeline...")
df_clean, audit_table = audit_and_clean_sensor_data(df_flat)
df_feat = engineer_physical_features(df_clean)

print("\n--- TABLE 1: DATA AUDIT & CLEANING SUMMARY ---")
print(audit_table.to_string(index=False))

split_comparison_rows = []
for skind in ["random", "time", "grouped"]:
    tr_idx, val_idx = split_sensor_data(df_feat, split_kind=skind, test_size=0.25, random_state=42)
    X_tr, y_tr = df_feat.loc[tr_idx, CORE_FEATURES].to_numpy(), df_feat.loc[tr_idx, TARGET_COL].to_numpy()
    X_va, y_va = df_feat.loc[val_idx, CORE_FEATURES].to_numpy(), df_feat.loc[val_idx, TARGET_COL].to_numpy()

    models = build_ml_regressors(random_state=42)
    for mname, pipe in models.items():
        _, fit_ms, fit_kb = profile_callable(pipe.fit, X_tr, y_tr)
        preds, inf_ms, inf_kb = profile_callable(pipe.predict, X_va)
        metrics = compute_engineering_metrics(y_va, preds)
        split_comparison_rows.append({
            "split_strategy": skind,
            "model": mname,
            "n_train": len(tr_idx),
            "n_val": len(val_idx),
            **metrics,
            "fit_time_ms": round(fit_ms, 2),
            "inference_time_ms": round(inf_ms, 2),
            "peak_memory_kb": round(max(fit_kb, inf_kb), 1),
            "theoretical_flops": estimate_inference_flops(mname, len(val_idx), len(CORE_FEATURES))
        })

df_split_comp = pd.DataFrame(split_comparison_rows)
print("\n--- TABLE 2: VALIDATION ACROSS SPLITTING STRATEGIES ---")
print(df_split_comp.round(3).to_string(index=False))

print("\nExecuting Dropout Gap Sensitivity Experiment (this may take ~20 seconds)...")
tr_idx, val_idx = split_sensor_data(df_feat, split_kind="time", test_size=0.25, random_state=42)
df_gap_exp = run_gap_duration_experiment(
    df_feat.iloc[tr_idx].copy(),
    df_feat.iloc[val_idx].copy(),
    gap_sizes_samples=[1, 2, 4, 6, 12, 24, 48, 72],
    n_gaps_per_size=25,
    random_state=42
)

print("\n--- TABLE 4: SENSOR DROPOUT SENSITIVITY ---")
cols_exp = ["gap_samples", "gap_duration_hours", "method", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]
print(df_gap_exp[cols_exp].round(3).to_string(index=False))
df_gap_exp.to_csv("MMA3001_Project/results/metrics/table4_dropout_gap_experiment.csv", index=False)



[1/4] Re-extracting and standardizing JSON telemetry...
Extracted 179,447 standardized records in 13.34 seconds.
[2/4] Saving 'cleaned_env_sensors.parquet'...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[3/4] Testing module imports...
IMPORT ERROR: No module named 'src'
Traceback (most recent call last):
  File "/tmp/ipykernel_2668/2369479763.py", line 76, in <cell line: 0>
    from src.preprocessing import audit_and_clean_sensor_data, engineer_physical_features, split_sensor_data, CORE_FEATURES, TARGET_COL
ModuleNotFoundError: No module named 'src'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_2668/2369479763.py", line 83, in <cell line: 0>
    sys.exit(1)
    ~~~~~~~~^^^
SystemExit: 1

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/ultratb.py", line 1101, in get_records
    re

TypeError: object of type 'NoneType' has no len()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Monash Branding Colors
MONASH_BLUE = '#006DAE'
MONASH_GREY = '#333333'
MONASH_ORANGE = '#D9381E'

print("Generating 4 Publication-Ready Figures for Report...")

df_split = pd.read_csv("MMA3001_Project/results/metrics/table2_split_and_model_benchmark.csv")
df_gap = pd.read_csv("MMA3001_Project/results/metrics/table4_dropout_gap_experiment.csv")
df_feat = pd.read_csv("MMA3001_Project/results/metrics/table3_feature_importance.csv")

# ---------------------------------------------------------
# Figure 1: Baseline vs ML Benchmark (Accuracy vs Runtime)
# ---------------------------------------------------------
plt.figure(figsize=(8, 5))
time_split = df_split[df_split["split_strategy"] == "time"]
for _, row in time_split.iterrows():
    color = MONASH_BLUE if "Regression" in row["model"] or "Forest" in row["model"] else MONASH_GREY
    plt.scatter(row["inference_time_ms"], row["RMSE_ppm"], label=row["model"], s=100, c=color)

plt.xscale('log')
plt.xlabel("Inference Runtime [ms, log scale]")
plt.ylabel("Test RMSE [ppm]")
plt.title("Model Trade-off: Accuracy vs Computational Cost\n(Chronological Hold-out Split)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("MMA3001_Project/results/figures/fig1_benchmark.png", dpi=300)
plt.close()

# ---------------------------------------------------------
# Figure 2: The Data Leakage Problem (Random vs Time Split)
# ---------------------------------------------------------
rf_splits = df_split[df_split["model"] == "Random Forest"]
plt.figure(figsize=(7, 4.5))
bars = plt.bar(rf_splits["split_strategy"], rf_splits["RMSE_ppm"], color=[MONASH_ORANGE, MONASH_BLUE, MONASH_GREY])
plt.ylabel("RMSE [ppm]")
plt.title("Effect of Data Leakage on Random Forest Generalisation")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1, round(yval, 1), ha='center', va='bottom')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("MMA3001_Project/results/figures/fig2_leakage.png", dpi=300)
plt.close()

# ---------------------------------------------------------
# Figure 3: Sensor Dropout Gap Sensitivity (Independent Exp)
# ---------------------------------------------------------
plt.figure(figsize=(8, 5))
for method in df_gap["method"].unique():
    subset = df_gap[df_gap["method"] == method]
    linestyle = '--' if "Interpolation" in method or "Spline" in method else '-'
    plt.plot(subset["gap_duration_hours"], subset["RMSE_ppm"], marker='o', linestyle=linestyle, label=method)

plt.xlabel("Contiguous Sensor Dropout Duration [hours]")
plt.ylabel("Reconstruction RMSE [ppm]")
plt.title("Sensitivity to Missing Sensor Block Duration\n(ML vs Numerical Interpolation)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("MMA3001_Project/results/figures/fig3_dropout_sensitivity.png", dpi=300)
plt.close()

# ---------------------------------------------------------
# Figure 4: Feature Importance (Physical Dynamics)
# ---------------------------------------------------------
df_feat = df_feat.sort_values("rf_importance", ascending=True).tail(8) # Top 8
plt.figure(figsize=(8, 4))
plt.barh(df_feat["feature"], df_feat["rf_importance"], color=MONASH_BLUE)
plt.xlabel("Gini Feature Importance (Random Forest)")
plt.title("Drivers of Indoor CO2 Soft-Sensing")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("MMA3001_Project/results/figures/fig4_feature_importance.png", dpi=300)
plt.close()

print("[SUCCESS] 4 High-Resolution Engineering Figures saved to 'MMA3001_Project/results/figures/'")


Generating 4 Publication-Ready Figures for Report...


FileNotFoundError: [Errno 2] No such file or directory: 'MMA3001_Project/results/metrics/table2_split_and_model_benchmark.csv'

In [ ]:
import os
import sys
import time
import json
import subprocess
import tracemalloc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Callable, Any
from scipy.interpolate import interp1d, CubicSpline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

print("="*80)
print("MMA3001 — ALL-IN-ONE MASTER EXECUTION PIPELINE")
print("="*80)

# -----------------------------------------------------------------------------
# STEP 1: CREATE DIRECTORY STRUCTURE
# -----------------------------------------------------------------------------
print("\n[1/7] Creating project directory hierarchy...")
folders = [
    "MMA3001_Project/src",
    "MMA3001_Project/tests",
    "MMA3001_Project/docs",
    "MMA3001_Project/results/figures",
    "MMA3001_Project/results/metrics",
    "MMA3001_Project/data"
]
for fld in folders:
    os.makedirs(fld, exist_ok=True)

# -----------------------------------------------------------------------------
# STEP 2: WRITE SOURCE MODULES (src/*.py)
# -----------------------------------------------------------------------------
print("[2/7] Writing modular Python files...")

# src/__init__.py
with open("MMA3001_Project/src/__init__.py", "w") as f:
    f.write('"""MMA3001 Smart Infrastructure Sensor Reconstruction Package."""\n')

# src/preprocessing.py
with open("MMA3001_Project/src/preprocessing.py", "w") as f:
    f.write('''"""Data auditing, physical validation, and leak-free splitting."""
import numpy as np
import pandas as pd
from typing import Tuple, List, Dict

PHYSICAL_BOUNDS = {
    "carbon_dioxide": (350.0, 5000.0),
    "temperature": (0.0, 50.0),
    "humidity": (0.0, 100.0),
    "pressure": (950.0, 1060.0),
    "particulate_matter_1": (0.0, 500.0),
    "particulate_matter_2.5": (0.0, 500.0),
    "particulate_matter_4": (0.0, 500.0),
    "particulate_matter_10": (0.0, 500.0),
}

CORE_FEATURES = [
    "temperature", "humidity", "pressure",
    "particulate_matter_1", "particulate_matter_2.5",
    "particulate_matter_4", "particulate_matter_10",
    "hour_sin", "hour_cos", "dT_dt", "dRH_dt"
]
TARGET_COL = "carbon_dioxide"

def audit_and_clean_sensor_data(df: pd.DataFrame, start_date="2024-05-01", end_date="2024-12-31"):
    data = df.copy()
    if "datetime" not in data.columns:
        data["datetime"] = pd.to_datetime(data["timestamp"], unit="s", utc=True)
    ts_utc = pd.to_datetime(data["timestamp"], unit="s", utc=True)
    valid_window = (ts_utc >= pd.Timestamp(start_date, tz="UTC")) & (ts_utc <= pd.Timestamp(end_date + " 23:59:59", tz="UTC"))
    data = data.loc[valid_window].sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)
    data = data.drop_duplicates(subset=["sensor_id", "timestamp"], keep="first").reset_index(drop=True)
    for col, (low, high) in PHYSICAL_BOUNDS.items():
        if col in data.columns:
            data.loc[~data[col].between(low, high) & data[col].notna(), col] = np.nan
    data = data.dropna(subset=[TARGET_COL, "temperature", "humidity", "pressure"]).reset_index(drop=True)
    summary = pd.DataFrame({
        "metric": ["clean_observations", "unique_sensors", "min_timestamp", "max_timestamp"],
        "value": [len(data), data["sensor_id"].nunique(), str(data["datetime"].min()), str(data["datetime"].max())]
    })
    return data, summary

def engineer_physical_features(df: pd.DataFrame):
    data = df.copy()
    dt = pd.to_datetime(data["timestamp"], unit="s", utc=True).dt.tz_convert("Australia/Melbourne")
    f_hour = dt.dt.hour + dt.dt.minute / 60.0
    data["hour_sin"] = np.sin(2.0 * np.pi * f_hour / 24.0)
    data["hour_cos"] = np.cos(2.0 * np.pi * f_hour / 24.0)
    data["dT_dt"] = 0.0
    data["dRH_dt"] = 0.0
    for _, grp in data.groupby("sensor_id"):
        idx = grp.index
        dt_h = grp["timestamp"].diff() / 3600.0
        val = (dt_h > 1e-4) & (dt_h <= 2.0)
        data.loc[idx, "dT_dt"] = np.nan_to_num(np.where(val, grp["temperature"].diff() / dt_h, 0.0))
        data.loc[idx, "dRH_dt"] = np.nan_to_num(np.where(val, grp["humidity"].diff() / dt_h, 0.0))
    return data

def split_sensor_data(df: pd.DataFrame, split_kind="time", test_size=0.25, random_state=42):
    n = len(df)
    idx = np.arange(n)
    if split_kind == "random":
        rng = np.random.default_rng(random_state)
        shf = rng.permutation(idx)
        nv = int(n * test_size)
        return shf[nv:], shf[:nv]
    elif split_kind == "grouped":
        val_sensor = df["sensor_id"].unique()[-1]
        mask = (df["sensor_id"] == val_sensor).to_numpy()
        return idx[~mask], idx[mask]
    elif split_kind == "time":
        tr_l, val_l = [], []
        for _, grp in df.groupby("sensor_id"):
            ord_i = grp.sort_values("timestamp").index.to_numpy()
            c = int(len(ord_i) * (1.0 - test_size))
            tr_l.append(ord_i[:c])
            val_l.append(ord_i[c:])
        return np.concatenate(tr_l), np.concatenate(val_l)
''')

# src/models.py
with open("MMA3001_Project/src/models.py", "w") as f:
    f.write('''"""Numerical interpolation and ML regression models."""
import numpy as np
from scipy.interpolate import interp1d, CubicSpline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

def interpolate_sensor_gaps(t_k, y_k, t_q, method="linear", clip_bounds=(350.0, 5000.0)):
    tk, yk, tq = np.asarray(t_k, float), np.asarray(y_k, float), np.asarray(t_q, float)
    if np.any(np.diff(tk) <= 0):
        raise ValueError("time_known must be strictly increasing.")
    if method == "linear":
        pred = interp1d(tk, yk, kind="linear", bounds_error=False, fill_value=(yk[0], yk[-1]))(tq)
    elif method == "cubic":
        pred = CubicSpline(tk, yk, bc_type="not-a-knot", extrapolate=True)(tq)
    else:
        raise ValueError("Unknown method")
    if clip_bounds is not None:
        pred = np.clip(pred, clip_bounds[0], clip_bounds[1])
    return np.asarray(pred, float)

def build_ml_regressors(random_state=42):
    return {
        "Mean Baseline": Pipeline([("imp", SimpleImputer(strategy="median")), ("m", DummyRegressor(strategy="mean"))]),
        "Ridge Regression": Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("m", Ridge(alpha=10.0, random_state=random_state))]),
        "Random Forest": Pipeline([("imp", SimpleImputer(strategy="median")), ("m", RandomForestRegressor(n_estimators=60, max_depth=12, min_samples_leaf=4, n_jobs=-1, random_state=random_state))])
    }

def estimate_inference_flops(model_name, n_samples, n_features):
    if model_name == "Mean Baseline": return int(n_samples)
    if model_name == "Linear Interpolation": return int(n_samples * 4)
    if model_name == "Cubic Spline": return int(n_samples * 7)
    if model_name == "Ridge Regression": return int(n_samples * (4 * n_features + 1))
    if model_name == "Random Forest": return int(n_samples * (60 * 13 + 1))
    return int(n_samples)
''')

# src/evaluation.py
with open("MMA3001_Project/src/evaluation.py", "w") as f:
    f.write('''"""Evaluation metrics and profiler."""
import time, tracemalloc
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def compute_engineering_metrics(y_true, y_pred):
    yt, yp = np.asarray(y_true, float), np.asarray(y_pred, float)
    res = yt - yp
    return {
        "MAE_ppm": float(mean_absolute_error(yt, yp)),
        "RMSE_ppm": float(root_mean_squared_error(yt, yp)),
        "R2": float(r2_score(yt, yp)),
        "Max_Error_ppm": float(np.max(np.abs(res)))
    }

def profile_callable(func, *args, **kwargs):
    tracemalloc.start()
    t0 = time.perf_counter()
    res = func(*args, **kwargs)
    elapsed = (time.perf_counter() - t0) * 1000.0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return res, float(elapsed), float(peak / 1024.0)
''')

# src/experiments.py
with open("MMA3001_Project/src/experiments.py", "w") as f:
    f.write('''"""Contiguous sensor dropout experiment."""
import numpy as np
import pandas as pd
from src.preprocessing import CORE_FEATURES, TARGET_COL
from src.models import interpolate_sensor_gaps, build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics, profile_callable

def run_gap_duration_experiment(train_df, test_df, gap_sizes_samples=[1, 3, 6, 12, 24, 48], n_gaps=15, random_state=42):
    X_tr = train_df[CORE_FEATURES].to_numpy()
    y_tr = train_df[TARGET_COL].to_numpy()
    pipes = build_ml_regressors(random_state)
    for p in pipes.values(): p.fit(X_tr, y_tr)

    ps = test_df["sensor_id"].value_counts().index[0]
    sub = test_df[test_df["sensor_id"] == ps].sort_values("timestamp").reset_index(drop=True)
    ts = sub["timestamp"].to_numpy(float)
    y_f = sub[TARGET_COL].to_numpy(float)
    X_f = sub[CORE_FEATURES].to_numpy(float)
    n_tot = len(sub)
    dt_m = float(np.median(np.diff(ts)) / 60.0)
    rng = np.random.default_rng(random_state)
    records = []

    for g_len in gap_sizes_samples:
        margin = max(10, g_len + 2)
        v_starts = np.arange(margin, n_tot - g_len - margin, g_len + 6)
        c_starts = rng.choice(v_starts, size=min(n_gaps, len(v_starts)), replace=False)
        mask = np.zeros(n_tot, bool)
        for s in c_starts: mask[s : s + g_len] = True

        t_k, y_k = ts[~mask], y_f[~mask]
        t_q, y_q = ts[mask], y_f[mask]
        X_q = X_f[mask]
        g_hours = round((g_len * dt_m) / 60.0, 2)

        # Linear
        pred_l, t_l, _ = profile_callable(interpolate_sensor_gaps, t_k, y_k, t_q, "linear", None)
        records.append({"gap_samples": g_len, "gap_duration_hours": g_hours, "method": "Linear Interpolation", **compute_engineering_metrics(y_q, pred_l), "inference_time_ms": t_l, "estimated_flops": estimate_inference_flops("Linear Interpolation", len(y_q), len(CORE_FEATURES))})

        # Cubic
        pred_c, t_c, _ = profile_callable(interpolate_sensor_gaps, t_k, y_k, t_q, "cubic", None)
        records.append({"gap_samples": g_len, "gap_duration_hours": g_hours, "method": "Cubic Spline", **compute_engineering_metrics(y_q, pred_c), "inference_time_ms": t_c, "estimated_flops": estimate_inference_flops("Cubic Spline", len(y_q), len(CORE_FEATURES))})

        # ML models
        for mname in ["Ridge Regression", "Random Forest"]:
            pred_m, t_m, _ = profile_callable(pipes[mname].predict, X_q)
            records.append({"gap_samples": g_len, "gap_duration_hours": g_hours, "method": mname, **compute_engineering_metrics(y_q, pred_m), "inference_time_ms": t_m, "estimated_flops": estimate_inference_flops(mname, len(y_q), len(CORE_FEATURES))})

    return pd.DataFrame(records)
''')

# tests/test_project.py
with open("MMA3001_Project/tests/test_project.py", "w") as f:
    f.write('''"""Unit tests for MMA3001 Project."""
import numpy as np, pandas as pd, pytest
from src.preprocessing import split_sensor_data
from src.models import interpolate_sensor_gaps, estimate_inference_flops
from src.evaluation import compute_engineering_metrics

def test_linear_interpolation_exact_on_line():
    tk = np.array([0.0, 10.0, 20.0, 30.0, 40.0])
    yk = 3.0 * tk + 400.0
    tq = np.array([5.0, 15.0, 25.0])
    ypred = interpolate_sensor_gaps(tk, yk, tq, "linear", None)
    assert np.allclose(ypred, 3.0 * tq + 400.0)

def test_cubic_spline_exact_on_cubic_polynomial():
    tk = np.linspace(0.0, 10.0, 11)
    yk = 420.0 + 5.0 * tk - 0.8 * (tk ** 2) + 0.05 * (tk ** 3)
    tq = np.array([2.5, 5.5, 7.5])
    ypred = interpolate_sensor_gaps(tk, yk, tq, "cubic", None)
    assert np.allclose(ypred, 420.0 + 5.0 * tq - 0.8 * (tq ** 2) + 0.05 * (tq ** 3))

def test_interpolation_rejects_duplicate_timestamps():
    with pytest.raises(ValueError, match="strictly increasing"):
        interpolate_sensor_gaps(np.array([1.0, 2.0, 2.0]), np.array([1, 2, 2]), np.array([1.5]))

def test_time_split_zero_leakage():
    df = pd.DataFrame({"sensor_id": ["S1"]*10, "timestamp": np.arange(10)})
    tr, val = split_sensor_data(df, "time", 0.3)
    assert df.loc[tr, "timestamp"].max() < df.loc[val, "timestamp"].min()

def test_metrics_accuracy():
    yt = np.array([400.0, 500.0, 600.0])
    yp = np.array([410.0, 490.0, 620.0])
    m = compute_engineering_metrics(yt, yp)
    assert np.isclose(m["MAE_ppm"], 40.0 / 3.0)
''')

# -----------------------------------------------------------------------------
# STEP 3: RUN PYTEST AND GENERATE PDOC HTML DOCS
# -----------------------------------------------------------------------------
print("\n[3/7] Running pytest unit tests and generating pdoc HTML...")
if "MMA3001_Project" not in sys.path:
    sys.path.insert(0, "MMA3001_Project")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "pdoc"], check=True)
py_res = subprocess.run([sys.executable, "-m", "pytest", "MMA3001_Project/tests/test_project.py", "-v"], capture_output=True, text=True)
print(py_res.stdout)
with open("MMA3001_Project/results/metrics/pytest_report.txt", "w") as f:
    f.write(py_res.stdout)

subprocess.run([sys.executable, "-m", "pdoc", "--math", "MMA3001_Project/src", "-o", "MMA3001_Project/docs"], check=True)
print("pdoc HTML generated successfully in MMA3001_Project/docs/")

# -----------------------------------------------------------------------------
# STEP 4: LOAD CLEAN DATA & RUN DATA PIPELINE
# -----------------------------------------------------------------------------
print("\n[4/7] Loading cleaned sensor data & running feature engineering...")
from src.preprocessing import audit_and_clean_sensor_data, engineer_physical_features, split_sensor_data, CORE_FEATURES, TARGET_COL
from src.models import build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics, profile_callable
from src.experiments import run_gap_duration_experiment

df_raw = pd.read_parquet("cleaned_env_sensors.parquet")
df_clean, table1_audit = audit_and_clean_sensor_data(df_raw)
df_feat = engineer_physical_features(df_clean)

# Save Table 1
table1_audit.to_csv("MMA3001_Project/results/metrics/table1_data_audit.csv", index=False)
print("Table 1 saved.")

# -----------------------------------------------------------------------------
# STEP 5: MODEL BENCHMARK & DATA LEAKAGE AUDIT (TABLE 2 & 3)
# -----------------------------------------------------------------------------
print("\n[5/7] Executing Model Benchmark and Leakage Audit (Week 5.6)...")
split_rows = []
for skind in ["random", "time", "grouped"]:
    tr_idx, val_idx = split_sensor_data(df_feat, split_kind=skind, test_size=0.25, random_state=42)
    X_tr, y_tr = df_feat.loc[tr_idx, CORE_FEATURES].to_numpy(), df_feat.loc[tr_idx, TARGET_COL].to_numpy()
    X_va, y_va = df_feat.loc[val_idx, CORE_FEATURES].to_numpy(), df_feat.loc[val_idx, TARGET_COL].to_numpy()
    models = build_ml_regressors(random_state=42)
    for mname, pipe in models.items():
        _, fit_ms, _ = profile_callable(pipe.fit, X_tr, y_tr)
        preds, inf_ms, _ = profile_callable(pipe.predict, X_va)
        metrics = compute_engineering_metrics(y_va, preds)
        split_rows.append({
            "split_strategy": skind,
            "model": mname,
            **metrics,
            "fit_time_ms": round(fit_ms, 2),
            "inference_time_ms": round(inf_ms, 2),
            "theoretical_flops": estimate_inference_flops(mname, len(val_idx), len(CORE_FEATURES))
        })

df_split_comp = pd.DataFrame(split_rows)
# Save Table 2
df_split_comp.to_csv("MMA3001_Project/results/metrics/table2_split_benchmark.csv", index=False)
print("\n--- TABLE 2: MODEL BENCHMARK & SPLITTING STRATEGY (LEAKAGE AUDIT) ---")
print(df_split_comp[["split_strategy", "model", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]].round(3).to_string(index=False))

# Train final models on Time Split for Table 3 Feature Importances
tr_idx, val_idx = split_sensor_data(df_feat, split_kind="time", test_size=0.25, random_state=42)
train_df = df_feat.iloc[tr_idx].copy()
test_df = df_feat.iloc[val_idx].copy()

final_pipes = build_ml_regressors(random_state=42)
for p in final_pipes.values():
    p.fit(train_df[CORE_FEATURES].to_numpy(), train_df[TARGET_COL].to_numpy())

df_feat_imp = pd.DataFrame({
    "feature": CORE_FEATURES,
    "rf_importance": final_pipes["Random Forest"].named_steps["m"].feature_importances_,
    "ridge_scaled_coef": final_pipes["Ridge Regression"].named_steps["m"].coef_
}).sort_values("rf_importance", ascending=False)
df_feat_imp.to_csv("MMA3001_Project/results/metrics/table3_feature_importance.csv", index=False)
print("\n--- TABLE 3: PHYSICAL FEATURE IMPORTANCES ---")
print(df_feat_imp.round(4).to_string(index=False))

# -----------------------------------------------------------------------------
# STEP 6: INDEPENDENT EXPERIMENT (DROPOUT GAPS TABLE 4)
# -----------------------------------------------------------------------------
print("\n[6/7] Running Independent Experiment: Sensor Dropout Gap Sweep...")
df_gap_exp = run_gap_duration_experiment(
    train_df, test_df,
    gap_sizes_samples=[1, 3, 6, 12, 24, 48],
    n_gaps=15,
    random_state=42
)
df_gap_exp.to_csv("MMA3001_Project/results/metrics/table4_dropout_experiment.csv", index=False)
print("\n--- TABLE 4: SENSOR DROPOUT DURATION SENSITIVITY ---")
cols_exp = ["gap_samples", "gap_duration_hours", "method", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]
print(df_gap_exp[cols_exp].round(3).to_string(index=False))

# -----------------------------------------------------------------------------
# STEP 7: GENERATE 4 PUBLICATION-READY FIGURES & ZIP REPO
# -----------------------------------------------------------------------------
print("\n[7/7] Generating Monash-styled Publication Figures...")
MONASH_BLUE = '#006DAE'
MONASH_ORANGE = '#D9381E'
MONASH_GREY = '#333333'

# Figure 1: Model Benchmark on Chronological Time Split
plt.figure(figsize=(8, 4.8))
time_split_data = df_split_comp[df_split_comp["split_strategy"] == "time"]
for _, row in time_split_data.iterrows():
    c = MONASH_BLUE if "Ridge" in row["model"] or "Forest" in row["model"] else MONASH_GREY
    plt.scatter(row["inference_time_ms"], row["RMSE_ppm"], label=row["model"], s=110, color=c)
plt.xscale('log')
plt.xlabel("Inference Runtime per Batch [ms, log scale]")
plt.ylabel("Test RMSE [ppm]")
plt.title("Computational Trade-off: Accuracy vs Inference Cost (Time Split)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("MMA3001_Project/results/figures/fig1_benchmark.png", dpi=300)
plt.close()

# Figure 2: The Data Leakage Problem (Week 5.6)
plt.figure(figsize=(7, 4.5))
rf_data = df_split_comp[df_split_comp["model"] == "Random Forest"]
bars = plt.bar(rf_data["split_strategy"], rf_data["RMSE_ppm"], color=[MONASH_ORANGE, MONASH_BLUE, MONASH_GREY])
plt.ylabel("Test RMSE [ppm]")
plt.title("Effect of Data Leakage on Model Generalisation (Random Forest)")
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, y + 1.0, f"{y:.1f} ppm", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("MMA3001_Project/results/figures/fig2_leakage.png", dpi=300)
plt.close()

# Figure 3: Sensor Dropout Gap Sensitivity
plt.figure(figsize=(8.5, 5))
for m in df_gap_exp["method"].unique():
    sub = df_gap_exp[df_gap_exp["method"] == m]
    ls = '--' if "Interpolation" in m or "Spline" in m else '-'
    plt.plot(sub["gap_duration_hours"], sub["RMSE_ppm"], marker='o', linestyle=ls, label=m)
plt.xlabel("Contiguous Sensor Dropout Duration [hours]")
plt.ylabel("Reconstruction RMSE [ppm]")
plt.title("Sensitivity to Missing Sensor Block Duration (Numerical vs ML)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("MMA3001_Project/results/figures/fig3_dropout_sensitivity.png", dpi=300)
plt.close()

# Figure 4: Feature Importance
plt.figure(figsize=(8, 4.2))
top_feat = df_feat_imp.head(8).sort_values("rf_importance", ascending=True)
plt.barh(top_feat["feature"], top_feat["rf_importance"], color=MONASH_BLUE)
plt.xlabel("Gini Feature Importance (Random Forest)")
plt.title("Physical Drivers of Indoor CO2 Concentration")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("MMA3001_Project/results/figures/fig4_feature_importance.png", dpi=300)
plt.close()

print("Figures saved successfully in MMA3001_Project/results/figures/")

# Create Zip Archive
subprocess.run(["zip", "-r", "-q", "MMA3001_Project.zip", "MMA3001_Project"])
print("\n" + "="*80)
print("[SUCCESS] MMA3001_Project.zip READY FOR SUBMISSION!")
print("="*80)

from google.colab import files
files.download("MMA3001_Project.zip")


MMA3001 — ALL-IN-ONE MASTER EXECUTION PIPELINE

[1/7] Creating project directory hierarchy...
[2/7] Writing modular Python files...

[3/7] Running pytest unit tests and generating pdoc HTML...
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collecting ... collected 0 items / 1 error

==================================== ERRORS ====================================
____________ ERROR collecting MMA3001_Project/tests/test_project.py ____________
ImportError while importing test module '/content/MMA3001_Project/tests/test_project.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^

ModuleNotFoundError: No module named 'src'

In [ ]:
import os
import sys
import time
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("="*80)
print("MMA3001 — AUTOMATED EXECUTION PIPELINE (PATH-STABILIZED)")
print("="*80)

# Set absolute project paths
ROOT_DIR = os.path.abspath("/content")
PROJECT_DIR = os.path.join(ROOT_DIR, "MMA3001_Project")

for sub in ["src", "tests", "docs", "results/figures", "results/metrics", "data"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

# Register project directory in Python path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = f"{PROJECT_DIR}:{os.environ.get('PYTHONPATH', '')}"

# -----------------------------------------------------------------------------
# 1. WRITE MODULAR SOURCE FILES
# -----------------------------------------------------------------------------
print("\n[1/6] Writing modular code files to MMA3001_Project/src/ ...")

# src/__init__.py
with open(os.path.join(PROJECT_DIR, "src/__init__.py"), "w") as f:
    f.write('"""MMA3001 Smart Infrastructure Sensor Reconstruction Package."""\n')

# src/preprocessing.py
with open(os.path.join(PROJECT_DIR, "src/preprocessing.py"), "w") as f:
    f.write('''"""Data auditing, physical domain checks, and leak-free splitting."""
import numpy as np
import pandas as pd
from typing import Tuple, List, Dict

PHYSICAL_BOUNDS = {
    "carbon_dioxide": (350.0, 5000.0),
    "temperature": (0.0, 50.0),
    "humidity": (0.0, 100.0),
    "pressure": (950.0, 1060.0),
    "particulate_matter_1": (0.0, 500.0),
    "particulate_matter_2.5": (0.0, 500.0),
    "particulate_matter_4": (0.0, 500.0),
    "particulate_matter_10": (0.0, 500.0),
}

CORE_FEATURES = [
    "temperature", "humidity", "pressure",
    "particulate_matter_1", "particulate_matter_2.5",
    "particulate_matter_4", "particulate_matter_10",
    "hour_sin", "hour_cos", "dT_dt", "dRH_dt"
]
TARGET_COL = "carbon_dioxide"

def audit_and_clean_sensor_data(df: pd.DataFrame, start_date="2024-05-01", end_date="2024-12-31"):
    data = df.copy()
    if "datetime" not in data.columns:
        data["datetime"] = pd.to_datetime(data["timestamp"], unit="s", utc=True)
    ts_utc = pd.to_datetime(data["timestamp"], unit="s", utc=True)
    valid_window = (ts_utc >= pd.Timestamp(start_date, tz="UTC")) & (ts_utc <= pd.Timestamp(end_date + " 23:59:59", tz="UTC"))
    data = data.loc[valid_window].sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)
    data = data.drop_duplicates(subset=["sensor_id", "timestamp"], keep="first").reset_index(drop=True)
    for col, (low, high) in PHYSICAL_BOUNDS.items():
        if col in data.columns:
            data.loc[~data[col].between(low, high) & data[col].notna(), col] = np.nan
    data = data.dropna(subset=[TARGET_COL, "temperature", "humidity", "pressure"]).reset_index(drop=True)
    summary = pd.DataFrame({
        "metric": ["clean_observations", "unique_sensors", "min_timestamp", "max_timestamp"],
        "value": [len(data), data["sensor_id"].nunique(), str(data["datetime"].min()), str(data["datetime"].max())]
    })
    return data, summary

def engineer_physical_features(df: pd.DataFrame):
    data = df.copy()
    dt = pd.to_datetime(data["timestamp"], unit="s", utc=True).dt.tz_convert("Australia/Melbourne")
    f_hour = dt.dt.hour + dt.dt.minute / 60.0
    data["hour_sin"] = np.sin(2.0 * np.pi * f_hour / 24.0)
    data["hour_cos"] = np.cos(2.0 * np.pi * f_hour / 24.0)
    data["dT_dt"] = 0.0
    data["dRH_dt"] = 0.0
    for _, grp in data.groupby("sensor_id"):
        idx = grp.index
        dt_h = grp["timestamp"].diff() / 3600.0
        val = (dt_h > 1e-4) & (dt_h <= 2.0)
        data.loc[idx, "dT_dt"] = np.nan_to_num(np.where(val, grp["temperature"].diff() / dt_h, 0.0))
        data.loc[idx, "dRH_dt"] = np.nan_to_num(np.where(val, grp["humidity"].diff() / dt_h, 0.0))
    return data

def split_sensor_data(df: pd.DataFrame, split_kind="time", test_size=0.25, random_state=42):
    n = len(df)
    idx = np.arange(n)
    if split_kind == "random":
        rng = np.random.default_rng(random_state)
        shf = rng.permutation(idx)
        nv = int(n * test_size)
        return shf[nv:], shf[:nv]
    elif split_kind == "grouped":
        val_sensor = df["sensor_id"].unique()[-1]
        mask = (df["sensor_id"] == val_sensor).to_numpy()
        return idx[~mask], idx[mask]
    elif split_kind == "time":
        tr_l, val_l = [], []
        for _, grp in df.groupby("sensor_id"):
            ord_i = grp.sort_values("timestamp").index.to_numpy()
            c = int(len(ord_i) * (1.0 - test_size))
            tr_l.append(ord_i[:c])
            val_l.append(ord_i[c:])
        return np.concatenate(tr_l), np.concatenate(val_l)
''')

# src/models.py
with open(os.path.join(PROJECT_DIR, "src/models.py"), "w") as f:
    f.write('''"""Numerical interpolation and ML regression models."""
import numpy as np
from scipy.interpolate import interp1d, CubicSpline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

def interpolate_sensor_gaps(t_k, y_k, t_q, method="linear", clip_bounds=(350.0, 5000.0)):
    tk, yk, tq = np.asarray(t_k, float), np.asarray(y_k, float), np.asarray(t_q, float)
    if np.any(np.diff(tk) <= 0):
        raise ValueError("time_known must be strictly increasing.")
    if method == "linear":
        pred = interp1d(tk, yk, kind="linear", bounds_error=False, fill_value=(yk[0], yk[-1]))(tq)
    elif method == "cubic":
        pred = CubicSpline(tk, yk, bc_type="not-a-knot", extrapolate=True)(tq)
    else:
        raise ValueError("Unknown method")
    if clip_bounds is not None:
        pred = np.clip(pred, clip_bounds[0], clip_bounds[1])
    return np.asarray(pred, float)

def build_ml_regressors(random_state=42):
    return {
        "Mean Baseline": Pipeline([("imp", SimpleImputer(strategy="median")), ("m", DummyRegressor(strategy="mean"))]),
        "Ridge Regression": Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("m", Ridge(alpha=10.0, random_state=random_state))]),
        "Random Forest": Pipeline([("imp", SimpleImputer(strategy="median")), ("m", RandomForestRegressor(n_estimators=60, max_depth=12, min_samples_leaf=4, n_jobs=-1, random_state=random_state))])
    }

def estimate_inference_flops(model_name, n_samples, n_features):
    if model_name == "Mean Baseline": return int(n_samples)
    if model_name == "Linear Interpolation": return int(n_samples * 4)
    if model_name == "Cubic Spline": return int(n_samples * 7)
    if model_name == "Ridge Regression": return int(n_samples * (4 * n_features + 1))
    if model_name == "Random Forest": return int(n_samples * (60 * 13 + 1))
    return int(n_samples)
''')

# src/evaluation.py
with open(os.path.join(PROJECT_DIR, "src/evaluation.py"), "w") as f:
    f.write('''"""Evaluation metrics and profiler."""
import time, tracemalloc
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def compute_engineering_metrics(y_true, y_pred):
    yt, yp = np.asarray(y_true, float), np.asarray(y_pred, float)
    res = yt - yp
    return {
        "MAE_ppm": float(mean_absolute_error(yt, yp)),
        "RMSE_ppm": float(root_mean_squared_error(yt, yp)),
        "R2": float(r2_score(yt, yp)),
        "Max_Error_ppm": float(np.max(np.abs(res)))
    }

def profile_callable(func, *args, **kwargs):
    tracemalloc.start()
    t0 = time.perf_counter()
    res = func(*args, **kwargs)
    elapsed = (time.perf_counter() - t0) * 1000.0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return res, float(elapsed), float(peak / 1024.0)
''')

# src/experiments.py
with open(os.path.join(PROJECT_DIR, "src/experiments.py"), "w") as f:
    f.write('''"""Contiguous sensor dropout experiment."""
import numpy as np
import pandas as pd
from src.preprocessing import CORE_FEATURES, TARGET_COL
from src.models import interpolate_sensor_gaps, build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics, profile_callable

def run_gap_duration_experiment(train_df, test_df, gap_sizes_samples=[1, 3, 6, 12, 24, 48], n_gaps=15, random_state=42):
    X_tr = train_df[CORE_FEATURES].to_numpy()
    y_tr = train_df[TARGET_COL].to_numpy()
    pipes = build_ml_regressors(random_state)
    for p in pipes.values(): p.fit(X_tr, y_tr)

    ps = test_df["sensor_id"].value_counts().index[0]
    sub = test_df[test_df["sensor_id"] == ps].sort_values("timestamp").reset_index(drop=True)
    ts = sub["timestamp"].to_numpy(float)
    y_f = sub[TARGET_COL].to_numpy(float)
    X_f = sub[CORE_FEATURES].to_numpy(float)
    n_tot = len(sub)
    dt_m = float(np.median(np.diff(ts)) / 60.0)
    rng = np.random.default_rng(random_state)
    records = []

    for g_len in gap_sizes_samples:
        margin = max(10, g_len + 2)
        v_starts = np.arange(margin, n_tot - g_len - margin, g_len + 6)
        c_starts = rng.choice(v_starts, size=min(n_gaps, len(v_starts)), replace=False)
        mask = np.zeros(n_tot, bool)
        for s in c_starts: mask[s : s + g_len] = True

        t_k, y_k = ts[~mask], y_f[~mask]
        t_q, y_q = ts[mask], y_f[mask]
        X_q = X_f[mask]
        g_hours = round((g_len * dt_m) / 60.0, 2)

        # Linear
        pred_l, t_l, _ = profile_callable(interpolate_sensor_gaps, t_k, y_k, t_q, "linear", None)
        records.append({"gap_samples": g_len, "gap_duration_hours": g_hours, "method": "Linear Interpolation", **compute_engineering_metrics(y_q, pred_l), "inference_time_ms": t_l, "estimated_flops": estimate_inference_flops("Linear Interpolation", len(y_q), len(CORE_FEATURES))})

        # Cubic
        pred_c, t_c, _ = profile_callable(interpolate_sensor_gaps, t_k, y_k, t_q, "cubic", None)
        records.append({"gap_samples": g_len, "gap_duration_hours": g_hours, "method": "Cubic Spline", **compute_engineering_metrics(y_q, pred_c), "inference_time_ms": t_c, "estimated_flops": estimate_inference_flops("Cubic Spline", len(y_q), len(CORE_FEATURES))})

        # ML models
        for mname in ["Ridge Regression", "Random Forest"]:
            pred_m, t_m, _ = profile_callable(pipes[mname].predict, X_q)
            records.append({"gap_samples": g_len, "gap_duration_hours": g_hours, "method": mname, **compute_engineering_metrics(y_q, pred_m), "inference_time_ms": t_m, "estimated_flops": estimate_inference_flops(mname, len(y_q), len(CORE_FEATURES))})

    return pd.DataFrame(records)
''')

# tests/test_project.py (with explicit path insertion)
with open(os.path.join(PROJECT_DIR, "tests/test_project.py"), "w") as f:
    f.write('''"""Unit tests for MMA3001 Project."""
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
import numpy as np, pandas as pd, pytest
from src.preprocessing import split_sensor_data
from src.models import interpolate_sensor_gaps, estimate_inference_flops
from src.evaluation import compute_engineering_metrics

def test_linear_interpolation_exact_on_line():
    tk = np.array([0.0, 10.0, 20.0, 30.0, 40.0])
    yk = 3.0 * tk + 400.0
    tq = np.array([5.0, 15.0, 25.0])
    ypred = interpolate_sensor_gaps(tk, yk, tq, "linear", None)
    assert np.allclose(ypred, 3.0 * tq + 400.0)

def test_cubic_spline_exact_on_cubic_polynomial():
    tk = np.linspace(0.0, 10.0, 11)
    yk = 420.0 + 5.0 * tk - 0.8 * (tk ** 2) + 0.05 * (tk ** 3)
    tq = np.array([2.5, 5.5, 7.5])
    ypred = interpolate_sensor_gaps(tk, yk, tq, "cubic", None)
    assert np.allclose(ypred, 420.0 + 5.0 * tq - 0.8 * (tq ** 2) + 0.05 * (tq ** 3))

def test_interpolation_rejects_duplicate_timestamps():
    with pytest.raises(ValueError, match="strictly increasing"):
        interpolate_sensor_gaps(np.array([1.0, 2.0, 2.0]), np.array([1, 2, 2]), np.array([1.5]))

def test_time_split_zero_leakage():
    df = pd.DataFrame({"sensor_id": ["S1"]*10, "timestamp": np.arange(10)})
    tr, val = split_sensor_data(df, "time", 0.3)
    assert df.loc[tr, "timestamp"].max() < df.loc[val, "timestamp"].min()

def test_metrics_accuracy():
    yt = np.array([400.0, 500.0, 600.0])
    yp = np.array([410.0, 490.0, 620.0])
    m = compute_engineering_metrics(yt, yp)
    assert np.isclose(m["MAE_ppm"], 40.0 / 3.0)
''')

# -----------------------------------------------------------------------------
# 2. RUN TESTS AND DOCS
# -----------------------------------------------------------------------------
print("\n[2/6] Running pytest and generating pdoc HTML...")
sub_env = os.environ.copy()
sub_env["PYTHONPATH"] = f"{PROJECT_DIR}:{sub_env.get('PYTHONPATH', '')}"

py_res = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_project.py", "-v"],
    cwd=PROJECT_DIR, capture_output=True, text=True, env=sub_env
)
print(py_res.stdout)
with open(os.path.join(PROJECT_DIR, "results/metrics/pytest_report.txt"), "w") as f:
    f.write(py_res.stdout)

subprocess.run(
    [sys.executable, "-m", "pdoc", "--math", "src", "-o", "docs"],
    cwd=PROJECT_DIR, check=True, env=sub_env
)
print("[OK] pdoc HTML generated in MMA3001_Project/docs/")

# -----------------------------------------------------------------------------
# 3. RUN MODEL PIPELINE & LEAKAGE AUDIT
# -----------------------------------------------------------------------------
print("\n[3/6] Running data pipeline, leakage audit, and model benchmarking...")
from src.preprocessing import audit_and_clean_sensor_data, engineer_physical_features, split_sensor_data, CORE_FEATURES, TARGET_COL
from src.models import build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics, profile_callable
from src.experiments import run_gap_duration_experiment

parquet_path = "/content/cleaned_env_sensors.parquet"
if not os.path.exists(parquet_path):
    parquet_path = "cleaned_env_sensors.parquet"

df_raw = pd.read_parquet(parquet_path)
df_clean, table1_audit = audit_and_clean_sensor_data(df_raw)
df_feat = engineer_physical_features(df_clean)

table1_audit.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table1_data_audit.csv"), index=False)

split_rows = []
for skind in ["random", "time", "grouped"]:
    tr_idx, val_idx = split_sensor_data(df_feat, split_kind=skind, test_size=0.25, random_state=42)
    X_tr, y_tr = df_feat.loc[tr_idx, CORE_FEATURES].to_numpy(), df_feat.loc[tr_idx, TARGET_COL].to_numpy()
    X_va, y_va = df_feat.loc[val_idx, CORE_FEATURES].to_numpy(), df_feat.loc[val_idx, TARGET_COL].to_numpy()
    models = build_ml_regressors(random_state=42)
    for mname, pipe in models.items():
        _, fit_ms, _ = profile_callable(pipe.fit, X_tr, y_tr)
        preds, inf_ms, _ = profile_callable(pipe.predict, X_va)
        metrics = compute_engineering_metrics(y_va, preds)
        split_rows.append({
            "split_strategy": skind,
            "model": mname,
            **metrics,
            "fit_time_ms": round(fit_ms, 2),
            "inference_time_ms": round(inf_ms, 2),
            "theoretical_flops": estimate_inference_flops(mname, len(val_idx), len(CORE_FEATURES))
        })

df_split_comp = pd.DataFrame(split_rows)
df_split_comp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table2_split_benchmark.csv"), index=False)
print("\n--- TABLE 2: MODEL BENCHMARK & DATA LEAKAGE AUDIT ---")
print(df_split_comp[["split_strategy", "model", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]].round(3).to_string(index=False))

# Feature importances
tr_idx, val_idx = split_sensor_data(df_feat, split_kind="time", test_size=0.25, random_state=42)
train_df = df_feat.iloc[tr_idx].copy()
test_df = df_feat.iloc[val_idx].copy()

final_pipes = build_ml_regressors(random_state=42)
for p in final_pipes.values():
    p.fit(train_df[CORE_FEATURES].to_numpy(), train_df[TARGET_COL].to_numpy())

df_feat_imp = pd.DataFrame({
    "feature": CORE_FEATURES,
    "rf_importance": final_pipes["Random Forest"].named_steps["m"].feature_importances_,
    "ridge_scaled_coef": final_pipes["Ridge Regression"].named_steps["m"].coef_
}).sort_values("rf_importance", ascending=False)
df_feat_imp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table3_feature_importance.csv"), index=False)
print("\n--- TABLE 3: PHYSICAL FEATURE IMPORTANCES ---")
print(df_feat_imp.round(4).to_string(index=False))

# -----------------------------------------------------------------------------
# 4. RUN DROPOUT GAP SENSITIVITY EXPERIMENT
# -----------------------------------------------------------------------------
print("\n[4/6] Running Independent Experiment: Sensor Dropout Gap Sweep...")
df_gap_exp = run_gap_duration_experiment(
    train_df, test_df,
    gap_sizes_samples=[1, 3, 6, 12, 24, 48],
    n_gaps=15,
    random_state=42
)
df_gap_exp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table4_dropout_experiment.csv"), index=False)
print("\n--- TABLE 4: SENSOR DROPOUT DURATION SENSITIVITY ---")
cols_exp = ["gap_samples", "gap_duration_hours", "method", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]
print(df_gap_exp[cols_exp].round(3).to_string(index=False))

# -----------------------------------------------------------------------------
# 5. GENERATE 4 PUBLICATION FIGURES
# -----------------------------------------------------------------------------
print("\n[5/6] Generating 4 Monash-styled Publication Figures...")
MONASH_BLUE = '#006DAE'
MONASH_ORANGE = '#D9381E'
MONASH_GREY = '#333333'

# Figure 1: Accuracy vs Runtime Tradeoff
plt.figure(figsize=(8, 4.8))
time_data = df_split_comp[df_split_comp["split_strategy"] == "time"]
for _, row in time_data.iterrows():
    c = MONASH_BLUE if "Ridge" in row["model"] or "Forest" in row["model"] else MONASH_GREY
    plt.scatter(row["inference_time_ms"], row["RMSE_ppm"], label=row["model"], s=110, color=c)
plt.xscale('log')
plt.xlabel("Inference Runtime per Batch [ms, log scale]")
plt.ylabel("Test RMSE [ppm]")
plt.title("Model Trade-off: Accuracy vs Computational Cost (Time Split)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig1_benchmark.png"), dpi=300)
plt.close()

# Figure 2: Data Leakage Demonstration
plt.figure(figsize=(7, 4.5))
rf_data = df_split_comp[df_split_comp["model"] == "Random Forest"]
bars = plt.bar(rf_data["split_strategy"], rf_data["RMSE_ppm"], color=[MONASH_ORANGE, MONASH_BLUE, MONASH_GREY])
plt.ylabel("Test RMSE [ppm]")
plt.title("Effect of Data Leakage on Generalisation Error (Random Forest)")
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, y + 1.0, f"{y:.1f} ppm", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig2_leakage.png"), dpi=300)
plt.close()

# Figure 3: Sensor Dropout Sensitivity
plt.figure(figsize=(8.5, 5))
for m in df_gap_exp["method"].unique():
    sub = df_gap_exp[df_gap_exp["method"] == m]
    ls = '--' if "Interpolation" in m or "Spline" in m else '-'
    plt.plot(sub["gap_duration_hours"], sub["RMSE_ppm"], marker='o', linestyle=ls, label=m)
plt.xlabel("Contiguous Sensor Dropout Duration [hours]")
plt.ylabel("Reconstruction RMSE [ppm]")
plt.title("Sensitivity to Missing Sensor Block Duration (Numerical vs ML)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig3_dropout_sensitivity.png"), dpi=300)
plt.close()

# Figure 4: Feature Importance
plt.figure(figsize=(8, 4.2))
top_feat = df_feat_imp.head(8).sort_values("rf_importance", ascending=True)
plt.barh(top_feat["feature"], top_feat["rf_importance"], color=MONASH_BLUE)
plt.xlabel("Gini Feature Importance (Random Forest)")
plt.title("Physical Drivers of Indoor CO2 Concentration")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig4_feature_importance.png"), dpi=300)
plt.close()
print("[OK] All 4 Figures saved to results/figures/")

# -----------------------------------------------------------------------------
# 6. ZIP AND DOWNLOAD ARCHIVE
# -----------------------------------------------------------------------------
print("\n[6/6] Zipping MMA3001_Project repository for Moodle submission...")
zip_path = os.path.join(ROOT_DIR, "MMA3001_Project.zip")
subprocess.run(["zip", "-r", "-q", zip_path, "MMA3001_Project"], cwd=ROOT_DIR, check=True)

print("="*80)
print(f"[SUCCESS] Download ready: {zip_path}")
print("="*80)

from google.colab import files
files.download(zip_path)


MMA3001 — AUTOMATED EXECUTION PIPELINE (PATH-STABILIZED)

[1/6] Writing modular code files to MMA3001_Project/src/ ...

[2/6] Running pytest and generating pdoc HTML...
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/MMA3001_Project
plugins: platformdirs-4.13.5, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collecting ... collected 5 items

tests/test_project.py::test_linear_interpolation_exact_on_line PASSED    [ 20%]
tests/test_project.py::test_cubic_spline_exact_on_cubic_polynomial PASSED [ 40%]
tests/test_project.py::test_interpolation_rejects_duplicate_timestamps PASSED [ 60%]
tests/test_project.py::test_time_split_zero_leakage PASSED               [ 80%]
tests/test_project.py::test_metrics_accuracy PASSED                      [100%]

============================== 5 passed in 1.42s ===============================

[OK] pdoc

ValueError: Found array with 0 sample(s) (shape=(0, 11)) while a minimum of 1 is required by SimpleImputer.

In [ ]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT_DIR = os.path.abspath("/content")
PROJECT_DIR = os.path.join(ROOT_DIR, "MMA3001_Project")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from src.preprocessing import (
    audit_and_clean_sensor_data,
    engineer_physical_features,
    split_sensor_data,
    CORE_FEATURES,
    TARGET_COL
)
from src.models import build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics, profile_callable
from src.experiments import run_gap_duration_experiment

# 1. Load data with full valid deployment window (May 2023 to 2026)
parquet_path = "/content/cleaned_env_sensors.parquet"
if not os.path.exists(parquet_path):
    parquet_path = "cleaned_env_sensors.parquet"

print("[1/5] Loading and cleaning sensor dataset...")
df_raw = pd.read_parquet(parquet_path)
df_clean, table1_audit = audit_and_clean_sensor_data(
    df_raw, start_date="2023-01-01", end_date="2026-12-31"
)
df_feat = engineer_physical_features(df_clean)

table1_audit.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table1_data_audit.csv"), index=False)
print(f"Dataset ready: {len(df_feat):,} clean observations across {df_feat['sensor_id'].nunique()} sensors.")

# 2. Benchmark Models & Data Leakage Audit across Splitting Strategies
print("\n[2/5] Running Model Benchmark & Data Leakage Audit (Week 5.6)...")
split_rows = []
for skind in ["random", "time", "grouped"]:
    tr_idx, val_idx = split_sensor_data(df_feat, split_kind=skind, test_size=0.25, random_state=42)
    X_tr = df_feat.iloc[tr_idx][CORE_FEATURES].to_numpy()
    y_tr = df_feat.iloc[tr_idx][TARGET_COL].to_numpy()
    X_va = df_feat.iloc[val_idx][CORE_FEATURES].to_numpy()
    y_va = df_feat.iloc[val_idx][TARGET_COL].to_numpy()

    models = build_ml_regressors(random_state=42)
    for mname, pipe in models.items():
        _, fit_ms, _ = profile_callable(pipe.fit, X_tr, y_tr)
        preds, inf_ms, _ = profile_callable(pipe.predict, X_va)
        metrics = compute_engineering_metrics(y_va, preds)
        split_rows.append({
            "split_strategy": skind,
            "model": mname,
            **metrics,
            "fit_time_ms": round(fit_ms, 2),
            "inference_time_ms": round(inf_ms, 2),
            "theoretical_flops": estimate_inference_flops(mname, len(val_idx), len(CORE_FEATURES))
        })

df_split_comp = pd.DataFrame(split_rows)
df_split_comp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table2_split_benchmark.csv"), index=False)
print("\n--- TABLE 2: MODEL BENCHMARK & SPLITTING STRATEGY (LEAKAGE AUDIT) ---")
print(df_split_comp[["split_strategy", "model", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]].round(3).to_string(index=False))

# 3. Compute Physical Feature Importances on Time Split
print("\n[3/5] Computing Feature Importances on Chronological Time Split...")
tr_idx, val_idx = split_sensor_data(df_feat, split_kind="time", test_size=0.25, random_state=42)
train_df = df_feat.iloc[tr_idx].copy()
test_df = df_feat.iloc[val_idx].copy()

final_pipes = build_ml_regressors(random_state=42)
for p in final_pipes.values():
    p.fit(train_df[CORE_FEATURES].to_numpy(), train_df[TARGET_COL].to_numpy())

df_feat_imp = pd.DataFrame({
    "feature": CORE_FEATURES,
    "rf_importance": final_pipes["Random Forest"].named_steps["m"].feature_importances_,
    "ridge_scaled_coef": final_pipes["Ridge Regression"].named_steps["m"].coef_
}).sort_values("rf_importance", ascending=False)
df_feat_imp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table3_feature_importance.csv"), index=False)
print("\n--- TABLE 3: PHYSICAL FEATURE IMPORTANCES ---")
print(df_feat_imp.round(4).to_string(index=False))

# 4. Independent Experiment: Sensor Dropout Duration Sensitivity Sweep
print("\n[4/5] Executing Independent Experiment: Sensor Dropout Gap Duration Sweep...")
df_gap_exp = run_gap_duration_experiment(
    train_df, test_df,
    gap_sizes_samples=[1, 3, 6, 12, 24, 48],
    n_gaps=15,
    random_state=42
)
df_gap_exp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table4_dropout_experiment.csv"), index=False)
print("\n--- TABLE 4: SENSOR DROPOUT DURATION SENSITIVITY ---")
cols_exp = ["gap_samples", "gap_duration_hours", "method", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]
print(df_gap_exp[cols_exp].round(3).to_string(index=False))

# 5. Generate Monash-Styled Publication Figures & Download Zip
print("\n[5/5] Generating Publication-Grade Figures & Packaging...")
MONASH_BLUE = '#006DAE'
MONASH_ORANGE = '#D9381E'
MONASH_GREY = '#333333'

# Figure 1: Accuracy vs Runtime
plt.figure(figsize=(8, 4.8))
time_data = df_split_comp[df_split_comp["split_strategy"] == "time"]
for _, row in time_data.iterrows():
    c = MONASH_BLUE if "Ridge" in row["model"] or "Forest" in row["model"] else MONASH_GREY
    plt.scatter(row["inference_time_ms"], row["RMSE_ppm"], label=row["model"], s=110, color=c)
plt.xscale('log')
plt.xlabel("Inference Runtime per Batch [ms, log scale]")
plt.ylabel("Test RMSE [ppm]")
plt.title("Computational Trade-off: Accuracy vs Inference Cost (Time Split)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig1_benchmark.png"), dpi=300)
plt.close()

# Figure 2: Data Leakage
plt.figure(figsize=(7, 4.5))
rf_data = df_split_comp[df_split_comp["model"] == "Random Forest"]
bars = plt.bar(rf_data["split_strategy"], rf_data["RMSE_ppm"], color=[MONASH_ORANGE, MONASH_BLUE, MONASH_GREY])
plt.ylabel("Test RMSE [ppm]")
plt.title("Effect of Data Leakage on Generalisation Error (Random Forest)")
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, y + 1.0, f"{y:.1f} ppm", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig2_leakage.png"), dpi=300)
plt.close()

# Figure 3: Dropout Sensitivity
plt.figure(figsize=(8.5, 5))
for m in df_gap_exp["method"].unique():
    sub = df_gap_exp[df_gap_exp["method"] == m]
    ls = '--' if "Interpolation" in m or "Spline" in m else '-'
    plt.plot(sub["gap_duration_hours"], sub["RMSE_ppm"], marker='o', linestyle=ls, label=m)
plt.xlabel("Contiguous Sensor Dropout Duration [hours]")
plt.ylabel("Reconstruction RMSE [ppm]")
plt.title("Sensitivity to Missing Sensor Block Duration (Numerical vs ML)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig3_dropout_sensitivity.png"), dpi=300)
plt.close()

# Figure 4: Feature Importance
plt.figure(figsize=(8, 4.2))
top_feat = df_feat_imp.head(8).sort_values("rf_importance", ascending=True)
plt.barh(top_feat["feature"], top_feat["rf_importance"], color=MONASH_BLUE)
plt.xlabel("Gini Feature Importance (Random Forest)")
plt.title("Physical Drivers of Indoor CO2 Concentration")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig4_feature_importance.png"), dpi=300)
plt.close()

# Create Zip Archive
zip_path = os.path.join(ROOT_DIR, "MMA3001_Project.zip")
subprocess.run(["zip", "-r", "-q", zip_path, "MMA3001_Project"], cwd=ROOT_DIR, check=True)

print("\n" + "="*80)
print(f"[SUCCESS] All figures, tables, and {zip_path} generated!")
print("="*80)

from google.colab import files
files.download(zip_path)


[1/5] Loading and cleaning sensor dataset...
Dataset ready: 42,843 clean observations across 5 sensors.

[2/5] Running Model Benchmark & Data Leakage Audit (Week 5.6)...

--- TABLE 2: MODEL BENCHMARK & SPLITTING STRATEGY (LEAKAGE AUDIT) ---
split_strategy            model  MAE_ppm  RMSE_ppm     R2  Max_Error_ppm  inference_time_ms
        random    Mean Baseline   23.547    34.140 -0.000        636.385               3.86
        random Ridge Regression   20.829    30.420  0.206        620.045               5.17
        random    Random Forest   13.266    21.588  0.600        581.556             189.92
          time    Mean Baseline   22.635    34.616 -0.005        934.722               4.91
          time Ridge Regression   24.848    34.844 -0.018        917.351               7.38
          time    Random Forest   20.547    32.958  0.089        897.252              74.73
       grouped    Mean Baseline   36.126    51.319 -0.696        204.895               3.29
       grouped Ridge Re

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import os
import sys
import time
import json
import glob
import zipfile
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# ENVIRONMENT SETUP & REPOSITORY PATH REGISTRATION
# =============================================================================

# Root working directory on Google Colab
ROOT_DIR = os.path.abspath("/content")
PROJECT_DIR = os.path.join(ROOT_DIR, "MMA3001_Project")

# Scaffold standard project directory hierarchy
for sub in ["src", "tests", "docs", "results/figures", "results/metrics", "data"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

# Register project directory in Python path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.environ["PYTHONPATH"] = f"{PROJECT_DIR}:{os.environ.get('PYTHONPATH', '')}"


# =============================================================================
# STEP 1: WRITE MODULAR SOURCE FILES
# =============================================================================
print("\n[1/6] Writing modular code files to MMA3001_Project/src/ ...")

# 1A. src/__init__.py
with open(os.path.join(PROJECT_DIR, "src/__init__.py"), "w") as f:
    f.write('"""MMA3001 Smart Infrastructure Sensor Reconstruction Package."""\n')

# 1B. src/preprocessing.py
with open(os.path.join(PROJECT_DIR, "src/preprocessing.py"), "w") as f:
    f.write('''"""
Data auditing, physical domain bounding, feature engineering, and leak-free splitting.
"""

import numpy as np
import pandas as pd
from typing import Tuple, List, Dict

PHYSICAL_BOUNDS: Dict[str, Tuple[float, float]] = {
    "carbon_dioxide": (350.0, 5000.0),
    "temperature": (0.0, 50.0),
    "humidity": (0.0, 100.0),
    "pressure": (950.0, 1060.0),
    "particulate_matter_1": (0.0, 500.0),
    "particulate_matter_2.5": (0.0, 500.0),
    "particulate_matter_4": (0.0, 500.0),
    "particulate_matter_10": (0.0, 500.0),
}

CORE_FEATURES: List[str] = [
    "temperature", "humidity", "pressure",
    "particulate_matter_1", "particulate_matter_2.5",
    "particulate_matter_4", "particulate_matter_10",
    "hour_sin", "hour_cos", "dT_dt", "dRH_dt"
]

TARGET_COL: str = "carbon_dioxide"


def audit_and_clean_sensor_data(
    df: pd.DataFrame,
    start_date: str = "2023-01-01",
    end_date: str = "2026-12-31"
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Audit raw sensor telemetry, remove duplicates, and enforce physical bounds."""
    required_cols = {"sensor_id", "timestamp", TARGET_COL}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"Input DataFrame missing required schema: {required_cols - set(df.columns)}")

    data = df.copy()
    if "datetime" not in data.columns:
        data["datetime"] = pd.to_datetime(data["timestamp"], unit="s", utc=True)

    ts_utc = pd.to_datetime(data["timestamp"], unit="s", utc=True)
    valid_window = (ts_utc >= pd.Timestamp(start_date, tz="UTC")) & (
        ts_utc <= pd.Timestamp(end_date + " 23:59:59", tz="UTC")
    )
    data = data.loc[valid_window].copy()
    data = data.sort_values(["sensor_id", "timestamp"]).reset_index(drop=True)
    data = data.drop_duplicates(subset=["sensor_id", "timestamp"], keep="first").reset_index(drop=True)

    for col, (low, high) in PHYSICAL_BOUNDS.items():
        if col in data.columns:
            invalid_mask = ~data[col].between(low, high) & data[col].notna()
            data.loc[invalid_mask, col] = np.nan

    essential_cols = [TARGET_COL, "temperature", "humidity", "pressure"]
    data = data.dropna(subset=essential_cols).reset_index(drop=True)

    summary = pd.DataFrame({
        "metric": ["clean_observations", "unique_sensors", "min_timestamp", "max_timestamp"],
        "value": [len(data), int(data["sensor_id"].nunique()), str(data["datetime"].min()), str(data["datetime"].max())]
    })
    return data, summary


def engineer_physical_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute cyclical diurnal time features and causal backward finite differences."""
    data = df.copy()
    dt_local = pd.to_datetime(data["timestamp"], unit="s", utc=True).dt.tz_convert("Australia/Melbourne")
    fractional_hour = dt_local.dt.hour + dt_local.dt.minute / 60.0

    data["hour_sin"] = np.sin(2.0 * np.pi * fractional_hour / 24.0)
    data["hour_cos"] = np.cos(2.0 * np.pi * fractional_hour / 24.0)
    data["dT_dt"] = 0.0
    data["dRH_dt"] = 0.0

    for _, group in data.groupby("sensor_id"):
        grp_idx = group.index
        dt_hours = group["timestamp"].diff() / 3600.0
        valid_dt = (dt_hours > 1e-4) & (dt_hours <= 2.0)
        rate_t = np.where(valid_dt, group["temperature"].diff() / dt_hours, 0.0)
        rate_rh = np.where(valid_dt, group["humidity"].diff() / dt_hours, 0.0)
        data.loc[grp_idx, "dT_dt"] = np.nan_to_num(rate_t, nan=0.0)
        data.loc[grp_idx, "dRH_dt"] = np.nan_to_num(rate_rh, nan=0.0)

    return data


def split_sensor_data(
    df: pd.DataFrame,
    split_kind: str = "time",
    test_size: float = 0.25,
    random_state: int = 42
) -> Tuple[np.ndarray, np.ndarray]:
    """Partition dataset indices using random, grouped, or chronological strategies."""
    if not (0.0 < test_size < 1.0):
        raise ValueError("test_size must be strictly between 0 and 1.")

    n_samples = len(df)
    indices = np.arange(n_samples)

    if split_kind == "random":
        rng = np.random.default_rng(random_state)
        shuffled = rng.permutation(indices)
        cutoff = int(n_samples * test_size)
        return shuffled[cutoff:], shuffled[:cutoff]

    elif split_kind == "grouped":
        sensors = df["sensor_id"].unique()
        if len(sensors) < 2:
            raise ValueError("Grouped split requires at least 2 distinct sensor units.")
        val_sensor = sensors[-1]
        val_mask = (df["sensor_id"] == val_sensor).to_numpy()
        return indices[~val_mask], indices[val_mask]

    elif split_kind == "time":
        train_idx_list, val_idx_list = [], []
        for _, group in df.groupby("sensor_id"):
            ordered_idx = group.sort_values("timestamp").index.to_numpy()
            cutoff = int(len(ordered_idx) * (1.0 - test_size))
            train_idx_list.append(ordered_idx[:cutoff])
            val_idx_list.append(ordered_idx[cutoff:])
        return np.concatenate(train_idx_list), np.concatenate(val_idx_list)

    else:
        raise ValueError(f"Unknown split_kind '{split_kind}'.")
''')

# -----------------------------------------------------------------------------
# 1C. src/models.py
# -----------------------------------------------------------------------------
with open(os.path.join(PROJECT_DIR, "src/models.py"), "w") as f:
    f.write('''"""
Numerical interpolators and machine-learning soft-sensor regression models.

This module provides 1D numerical interpolation routines (Piecewise Linear and
Natural Cubic Splines) and Scikit-Learn multivariate regression pipelines
(Baseline, Ridge, and Random Forest) for reconstructing missing environmental telemetry.
"""

import numpy as np
from scipy.interpolate import interp1d, CubicSpline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from typing import Dict, Tuple, Optional


def interpolate_sensor_gaps(
    time_known: np.ndarray,
    y_known: np.ndarray,
    time_query: np.ndarray,
    method: str = "linear",
    clip_bounds: Optional[Tuple[float, float]] = (350.0, 5000.0)
) -> np.ndarray:
    """
    Reconstruct missing sensor values using 1D temporal interpolation.

    Parameters
    ----------
    time_known : numpy.ndarray
        Strictly increasing 1D array of timestamps corresponding to known sensor readings [s].
    y_known : numpy.ndarray
        1D array of target sensor measurements recorded at `time_known`.
    time_query : numpy.ndarray
        1D array of query timestamps falling within simulated or real telemetry outages [s].
    method : {'linear', 'cubic'}, optional
        Numerical formulation to employ, by default 'linear'.
        - 'linear': Piecewise Linear interpolation with constant boundary fill.
        - 'cubic': Natural Cubic Spline with not-a-knot boundary conditions.
    clip_bounds : tuple of (float, float) or None, optional
        Physical operating bounds [low, high] used to clamp unanchored spline oscillations,
        by default (350.0, 5000.0). Set to None to expose unclipped polynomial overshoot.

    Returns
    -------
    y_pred : numpy.ndarray
        1D array of reconstructed target estimations evaluated at `time_query`.

    Raises
    ------
    ValueError
        If `time_known` contains non-increasing or duplicate timestamps, if array lengths
        mismatch, or if an unsupported interpolation method string is supplied.

    Notes
    -----
    Cubic splines are vulnerable to the Runge phenomenon over multi-hour gaps ($>2$ hours),
    where lack of intermediate interior knots produces extreme boundary oscillations.
    """
    tk = np.asarray(time_known, dtype=float)
    yk = np.asarray(y_known, dtype=float)
    tq = np.asarray(time_query, dtype=float)

    if np.any(np.diff(tk) <= 0):
        raise ValueError("time_known must be strictly increasing with no duplicates.")
    if tk.size != yk.size:
        raise ValueError("time_known and y_known arrays must have identical length.")

    if method == "linear":
        interpolator = interp1d(tk, yk, kind="linear", bounds_error=False, fill_value=(yk[0], yk[-1]))
        pred = interpolator(tq)
    elif method == "cubic":
        cs = CubicSpline(tk, yk, bc_type="not-a-knot", extrapolate=True)
        pred = cs(tq)
    else:
        raise ValueError(f"Unknown interpolation method: {method}")

    if clip_bounds is not None:
        pred = np.clip(pred, clip_bounds[0], clip_bounds[1])

    return np.asarray(pred, dtype=float)


def build_ml_regressors(random_state: int = 42) -> Dict[str, Pipeline]:
    """
    Construct standardized, leak-free Scikit-Learn regression pipelines.

    Parameters
    ----------
    random_state : int, optional
        Random seed ensuring deterministic tree splits and reproducibility, by default 42.

    Returns
    -------
    pipelines : dict of str to sklearn.pipeline.Pipeline
        Configured estimators wrapped with median imputers and feature scalers:
        - 'Mean Baseline': Uninformative baseline predicting the training sample mean.
        - 'Ridge Regression': L2-regularized linear model with standard scaling.
        - 'Random Forest': Ensemble of 60 trees constrained to maximum depth 12.

    Notes
    -----
    Data transformations are chained within Pipeline objects to prevent feature
    leakage between cross-validation folds and holdout splits.
    """
    return {
        "Mean Baseline": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m", DummyRegressor(strategy="mean"))
        ]),
        "Ridge Regression": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
            ("m", Ridge(alpha=10.0, random_state=random_state))
        ]),
        "Random Forest": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m", RandomForestRegressor(
                n_estimators=60,
                max_depth=12,
                min_samples_leaf=4,
                n_jobs=-1,
                random_state=random_state
            ))
        ])
    }


def estimate_inference_flops(model_name: str, n_samples: int, n_features: int) -> int:
    """
    Estimate theoretical floating-point operations (FLOPs) required for batch inference.

    Parameters
    ----------
    model_name : str
        Algorithm name ('Mean Baseline', 'Linear Interpolation', 'Cubic Spline',
        'Ridge Regression', or 'Random Forest').
    n_samples : int
        Number of query inference observations ($N$).
    n_features : int
        Number of input measurement channels ($p$).

    Returns
    -------
    flops : int
        Estimated total floating-point arithmetic operations across $N$ samples.

    Raises
    ------
    ValueError
        If `n_samples` is non-positive or `model_name` is unrecognized.
    """
    if n_samples <= 0:
        raise ValueError("n_samples must be positive.")

    if model_name == "Mean Baseline":
        return int(n_samples)
    elif model_name == "Linear Interpolation":
        return int(n_samples * 4)
    elif model_name == "Cubic Spline":
        return int(n_samples * 7)
    elif model_name == "Ridge Regression":
        return int(n_samples * (4 * n_features + 1))
    elif model_name == "Random Forest":
        return int(n_samples * (60 * 13 + 1))
    return int(n_samples)
''')
# 1D. src/evaluation.py
with open(os.path.join(PROJECT_DIR, "src/evaluation.py"), "w") as f:
    f.write('''"""
Engineering validation metrics and performance profiling utilities.
"""

import time
import tracemalloc
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from typing import Callable, Tuple, Any, Dict


def compute_engineering_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Compute quantitative engineering accuracy and consequence metrics."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    if yt.shape != yp.shape or yt.size == 0:
        raise ValueError("Arrays must be non-empty and have matching shapes.")
    residuals = yt - yp
    return {
        "MAE_ppm": float(mean_absolute_error(yt, yp)),
        "RMSE_ppm": float(root_mean_squared_error(yt, yp)),
        "R2": float(r2_score(yt, yp)),
        "Max_Error_ppm": float(np.max(np.abs(residuals)))
    }


def profile_callable(func: Callable, *args, **kwargs) -> Tuple[Any, float, float]:
    """Profile wall-clock execution time and peak memory allocation."""
    tracemalloc.start()
    t0 = time.perf_counter()
    res = func(*args, **kwargs)
    elapsed = (time.perf_counter() - t0) * 1000.0
    _, peak_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return res, float(elapsed), float(peak_bytes / 1024.0)
''')

# 1E. src/experiments.py
with open(os.path.join(PROJECT_DIR, "src/experiments.py"), "w") as f:
    f.write('''"""
Controlled experimental sensitivity investigation: Contiguous sensor dropout gaps.
"""

import numpy as np
import pandas as pd
from typing import List
from src.preprocessing import CORE_FEATURES, TARGET_COL
from src.models import interpolate_sensor_gaps, build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics, profile_callable


def run_gap_duration_experiment(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_sizes_samples: List[int] = [1, 3, 6, 12, 24, 48],
    n_gaps: int = 15,
    random_state: int = 42
) -> pd.DataFrame:
    """Measure reconstruction accuracy across escalating contiguous sensor dropout durations."""
    x_tr = train_df[CORE_FEATURES].to_numpy()
    y_tr = train_df[TARGET_COL].to_numpy()
    pipes = build_ml_regressors(random_state)
    for model in pipes.values():
        model.fit(x_tr, y_tr)

    primary_sensor = test_df["sensor_id"].value_counts().index[0]
    sub = test_df[test_df["sensor_id"] == primary_sensor].sort_values("timestamp").reset_index(drop=True)

    ts = sub["timestamp"].to_numpy(dtype=float)
    y_full = sub[TARGET_COL].to_numpy(dtype=float)
    x_full = sub[CORE_FEATURES].to_numpy(dtype=float)
    n_total = len(sub)

    median_dt_min = float(np.median(np.diff(ts)) / 60.0)
    rng = np.random.default_rng(random_state)
    records = []

    for g_len in gap_sizes_samples:
        margin = max(10, g_len + 2)
        valid_starts = np.arange(margin, n_total - g_len - margin, g_len + 6)
        chosen_starts = rng.choice(valid_starts, size=min(n_gaps, len(valid_starts)), replace=False)

        mask = np.zeros(n_total, dtype=bool)
        for s in chosen_starts:
            mask[s : s + g_len] = True

        t_known, y_known = ts[~mask], y_full[~mask]
        t_query, y_true_gap = ts[mask], y_full[mask]
        x_query = x_full[mask]
        g_hours = round((g_len * median_dt_min) / 60.0, 2)

        pred_lin, t_lin, _ = profile_callable(interpolate_sensor_gaps, t_known, y_known, t_query, "linear", None)
        records.append({
            "gap_samples": g_len,
            "gap_duration_hours": g_hours,
            "method": "Linear Interpolation",
            **compute_engineering_metrics(y_true_gap, pred_lin),
            "inference_time_ms": t_lin,
            "estimated_flops": estimate_inference_flops("Linear Interpolation", len(y_true_gap), len(CORE_FEATURES))
        })

        pred_cub, t_cub, _ = profile_callable(interpolate_sensor_gaps, t_known, y_known, t_query, "cubic", None)
        records.append({
            "gap_samples": g_len,
            "gap_duration_hours": g_hours,
            "method": "Cubic Spline",
            **compute_engineering_metrics(y_true_gap, pred_cub),
            "inference_time_ms": t_cub,
            "estimated_flops": estimate_inference_flops("Cubic Spline", len(y_true_gap), len(CORE_FEATURES))
        })

        for mname in ["Ridge Regression", "Random Forest"]:
            pred_ml, t_ml, _ = profile_callable(pipes[mname].predict, x_query)
            records.append({
                "gap_samples": g_len,
                "gap_duration_hours": g_hours,
                "method": mname,
                **compute_engineering_metrics(y_true_gap, pred_ml),
                "inference_time_ms": t_ml,
                "estimated_flops": estimate_inference_flops(mname, len(y_true_gap), len(CORE_FEATURES))
            })

    return pd.DataFrame(records)
''')

# 1F. tests/test_project.py
with open(os.path.join(PROJECT_DIR, "tests/test_project.py"), "w") as f:
    f.write('''"""
Automated unit verification test suite for MMA3001 sensor package.
"""

import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import numpy as np
import pandas as pd
import pytest
from src.preprocessing import split_sensor_data
from src.models import interpolate_sensor_gaps
from src.evaluation import compute_engineering_metrics


def test_linear_interpolation_exact_on_line():
    tk = np.array([0.0, 10.0, 20.0, 30.0, 40.0])
    yk = 3.0 * tk + 400.0
    tq = np.array([5.0, 15.0, 25.0])
    ypred = interpolate_sensor_gaps(tk, yk, tq, "linear", None)
    assert np.allclose(ypred, 3.0 * tq + 400.0)


def test_cubic_spline_exact_on_cubic_polynomial():
    tk = np.linspace(0.0, 10.0, 11)
    yk = 420.0 + 5.0 * tk - 0.8 * (tk ** 2) + 0.05 * (tk ** 3)
    tq = np.array([2.5, 5.5, 7.5])
    ypred = interpolate_sensor_gaps(tk, yk, tq, "cubic", None)
    assert np.allclose(ypred, 420.0 + 5.0 * tq - 0.8 * (tq ** 2) + 0.05 * (tq ** 3))


def test_interpolation_rejects_duplicate_timestamps():
    with pytest.raises(ValueError, match="strictly increasing"):
        interpolate_sensor_gaps(np.array([1.0, 2.0, 2.0]), np.array([1.0, 2.0, 2.0]), np.array([1.5]))


def test_time_split_zero_leakage():
    df = pd.DataFrame({"sensor_id": ["S1"] * 10, "timestamp": np.arange(10)})
    tr_idx, val_idx = split_sensor_data(df, "time", 0.3)
    assert df.loc[tr_idx, "timestamp"].max() < df.loc[val_idx, "timestamp"].min()


def test_metrics_accuracy():
    yt = np.array([400.0, 500.0, 600.0])
    yp = np.array([410.0, 490.0, 620.0])
    m = compute_engineering_metrics(yt, yp)
    assert np.isclose(m["MAE_ppm"], 40.0 / 3.0)
''')


# =============================================================================
# STEP 2: RUN AUTOMATED PYTEST SUITE & COMPILE PDOC DOCUMENTATION
# =============================================================================
print("\n[2/6] Installing dependencies, running pytest, and generating docs...")

# 1. Install pytest-html along with pytest and pdoc
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pytest", "pytest-html", "pdoc"],
    check=True
)

sub_env = os.environ.copy()
sub_env["PYTHONPATH"] = f"{PROJECT_DIR}:{sub_env.get('PYTHONPATH', '')}"

# 2. Run pytest with --html flag to export a standalone interactive HTML report
html_report_path = os.path.join(PROJECT_DIR, "results/metrics/pytest_report.html")
py_res = subprocess.run(
    [
        sys.executable, "-m", "pytest", "tests/test_project.py", "-v",
        f"--html={html_report_path}",
        "--self-contained-html"
    ],
    cwd=PROJECT_DIR, capture_output=True, text=True, env=sub_env
)
print(py_res.stdout)

# Keep the text summary backup as well
with open(os.path.join(PROJECT_DIR, "results/metrics/pytest_report.txt"), "w") as f:
    f.write(py_res.stdout)

print(f"[OK] Interactive pytest HTML report saved to: {html_report_path}")

# 3. Generate pdoc documentation with TeX math rendering support
pdoc_res = subprocess.run(
    [sys.executable, "-m", "pdoc", "--math", "src", "-o", "docs"],
    cwd=PROJECT_DIR, capture_output=True, text=True, env=sub_env
)

if pdoc_res.returncode == 0:
    print("[OK] pdoc HTML API reference generated in MMA3001_Project/docs/")
else:
    print(f"[WARNING] pdoc encountered an issue:\n{pdoc_res.stderr}")


# =============================================================================
# STEP 3: DATA INGESTION & PIPELINE PREPARATION
# =============================================================================
print("\n[3/6] Running data pipeline, leakage audit, and model benchmarking...")

parquet_path = "/content/cleaned_env_sensors.parquet"
if not os.path.exists(parquet_path):
    parquet_path = "cleaned_env_sensors.parquet"

# Reconstruct parquet if not found
if not os.path.exists(parquet_path):
    print("[INFO] 'cleaned_env_sensors.parquet' not found. Checking raw CSV files...")
    csv_candidates = glob.glob("env_data_extracted/**/*.csv", recursive=True) + glob.glob("*.csv")
    csv_candidates = [f for f in csv_candidates if "sample_data" not in f]

    # If no CSV is extracted, check for the raw zip archive
    if not csv_candidates:
        zip_targets = ["5EnvSensor_MayToDec2024_180kRows.zip"] + glob.glob("*.zip")
        valid_zips = [z for z in zip_targets if os.path.exists(z) and "MMA3001" not in z]
        if valid_zips:
            print(f"[INFO] Extracting {valid_zips[0]}...")
            with zipfile.ZipFile(valid_zips[0], 'r') as ref:
                ref.extractall("env_data_extracted")
            csv_candidates = glob.glob("env_data_extracted/**/*.csv", recursive=True) + glob.glob("*.csv")
            csv_candidates = [f for f in csv_candidates if "sample_data" not in f]
        else:
            raise FileNotFoundError(
                "\n[ERROR] Raw sensor zip file not found in /content/!\n"
                "Please upload '5EnvSensor_MayToDec2024_180kRows.zip' to the Colab files sidebar and re-run."
            )

    raw_csv_path = csv_candidates[0]
    print(f"[INFO] Reading {raw_csv_path} and parsing JSON telemetry...")
    df_raw_csv = pd.read_csv(raw_csv_path, low_memory=False)

    records = []
    for s_id, j_str in zip(df_raw_csv['sensorid'], df_raw_csv['jsondata']):
        try:
            data_list = json.loads(j_str)
            row_dict = {'sensor_id': str(s_id)}
            for item in data_list:
                v_meta = item.get('variable', {})
                v_name = v_meta.get('name', 'unknown').strip()
                vals = item.get('values', [])
                if vals:
                    if 'timestamp' not in row_dict:
                        row_dict['timestamp'] = vals[0].get('time')
                    row_dict[v_name] = vals[0].get('value')
            records.append(row_dict)
        except Exception:
            continue

    df_flat = pd.DataFrame(records)
    df_flat['datetime'] = pd.to_datetime(df_flat['timestamp'], unit='s', utc=True).dt.tz_convert('Australia/Melbourne')
    df_flat = df_flat.sort_values(by=['sensor_id', 'datetime']).reset_index(drop=True)
    df_flat.columns = [c.lower().replace(' ', '_').replace('-', '_') for c in df_flat.columns]
    df_flat.to_parquet(parquet_path, index=False)
    print(f"[OK] Successfully built and saved '{parquet_path}'.")

from src.preprocessing import (
    audit_and_clean_sensor_data,
    engineer_physical_features,
    split_sensor_data,
    CORE_FEATURES,
    TARGET_COL
)
from src.models import build_ml_regressors, estimate_inference_flops
from src.evaluation import compute_engineering_metrics, profile_callable
from src.experiments import run_gap_duration_experiment

# Load and clean telemetry
df_raw = pd.read_parquet(parquet_path)
df_clean, table1_audit = audit_and_clean_sensor_data(
    df_raw, start_date="2023-01-01", end_date="2026-12-31"
)
df_feat = engineer_physical_features(df_clean)
table1_audit.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table1_data_audit.csv"), index=False)
print(f"Dataset ready: {len(df_feat):,} clean observations across {df_feat['sensor_id'].nunique()} sensors.")

# Systematic Model Benchmark across Splitting Strategies (Table 2)
split_rows = []
for skind in ["random", "time", "grouped"]:
    tr_idx, val_idx = split_sensor_data(df_feat, split_kind=skind, test_size=0.25, random_state=42)
    x_tr = df_feat.iloc[tr_idx][CORE_FEATURES].to_numpy()
    y_tr = df_feat.iloc[tr_idx][TARGET_COL].to_numpy()
    x_va = df_feat.iloc[val_idx][CORE_FEATURES].to_numpy()
    y_va = df_feat.iloc[val_idx][TARGET_COL].to_numpy()

    models = build_ml_regressors(random_state=42)
    for mname, pipe in models.items():
        _, fit_ms, _ = profile_callable(pipe.fit, x_tr, y_tr)
        preds, inf_ms, _ = profile_callable(pipe.predict, x_va)
        metrics = compute_engineering_metrics(y_va, preds)
        split_rows.append({
            "split_strategy": skind,
            "model": mname,
            **metrics,
            "fit_time_ms": round(fit_ms, 2),
            "inference_time_ms": round(inf_ms, 2),
            "theoretical_flops": estimate_inference_flops(mname, len(val_idx), len(CORE_FEATURES))
        })

df_split_comp = pd.DataFrame(split_rows)
df_split_comp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table2_split_benchmark.csv"), index=False)
print("\n--- TABLE 2: MODEL BENCHMARK & DATA LEAKAGE AUDIT ---")
print(df_split_comp[["split_strategy", "model", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]].round(3).to_string(index=False))


# =============================================================================
# STEP 4: PHYSICAL FEATURE IMPORTANCE ANALYSIS (TABLE 3)
# =============================================================================
print("\n[4/6] Computing Feature Importances on Chronological Time Split...")

tr_idx, val_idx = split_sensor_data(df_feat, split_kind="time", test_size=0.25, random_state=42)
train_df = df_feat.iloc[tr_idx].copy()
test_df = df_feat.iloc[val_idx].copy()

final_pipes = build_ml_regressors(random_state=42)
for pipe in final_pipes.values():
    pipe.fit(train_df[CORE_FEATURES].to_numpy(), train_df[TARGET_COL].to_numpy())

df_feat_imp = pd.DataFrame({
    "feature": CORE_FEATURES,
    "rf_importance": final_pipes["Random Forest"].named_steps["m"].feature_importances_,
    "ridge_scaled_coef": final_pipes["Ridge Regression"].named_steps["m"].coef_
}).sort_values("rf_importance", ascending=False)

df_feat_imp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table3_feature_importance.csv"), index=False)
print("\n--- TABLE 3: PHYSICAL FEATURE IMPORTANCES ---")
print(df_feat_imp.round(4).to_string(index=False))


# =============================================================================
# STEP 5: SENSOR DROPOUT SENSITIVITY EXPERIMENT (TABLE 4)
# =============================================================================
print("\n[5/6] Executing Independent Experiment: Sensor Dropout Gap Sweep...")

df_gap_exp = run_gap_duration_experiment(
    train_df, test_df,
    gap_sizes_samples=[1, 3, 6, 12, 24, 48],
    n_gaps=15,
    random_state=42
)

df_gap_exp.to_csv(os.path.join(PROJECT_DIR, "results/metrics/table4_dropout_experiment.csv"), index=False)
print("\n--- TABLE 4: SENSOR DROPOUT DURATION SENSITIVITY ---")
cols_exp = ["gap_samples", "gap_duration_hours", "method", "MAE_ppm", "RMSE_ppm", "R2", "Max_Error_ppm", "inference_time_ms"]
print(df_gap_exp[cols_exp].round(3).to_string(index=False))


# =============================================================================
# STEP 6: PUBLICATION FIGURES & PACKAGING
# =============================================================================
print("\n[6/6] Generating Publication-Grade Figures & Packaging Archive...")

MONASH_BLUE = '#006DAE'
MONASH_ORANGE = '#D9381E'
MONASH_GREY = '#333333'

# Figure 1: Accuracy vs Inference Runtime Trade-off (Time Split)
plt.figure(figsize=(8, 4.8))
time_data = df_split_comp[df_split_comp["split_strategy"] == "time"]
for _, row in time_data.iterrows():
    c = MONASH_BLUE if "Ridge" in row["model"] or "Forest" in row["model"] else MONASH_GREY
    plt.scatter(row["inference_time_ms"], row["RMSE_ppm"], label=row["model"], s=110, color=c)
plt.xscale('log')
plt.xlabel("Inference Runtime per Batch [ms, log scale]")
plt.ylabel("Test RMSE [ppm]")
plt.title("Model Trade-off: Accuracy vs Computational Cost (Time Split)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig1_benchmark.png"), dpi=300)
plt.close()

# Figure 2: Demonstration of Temporal Data Leakage Error
plt.figure(figsize=(7, 4.5))
rf_data = df_split_comp[df_split_comp["model"] == "Random Forest"]
bars = plt.bar(rf_data["split_strategy"], rf_data["RMSE_ppm"], color=[MONASH_ORANGE, MONASH_BLUE, MONASH_GREY])
plt.ylabel("Test RMSE [ppm]")
plt.title("Effect of Data Leakage on Generalisation Error (Random Forest)")
for bar in bars:
    y_val = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, y_val + 1.0, f"{y_val:.1f} ppm", ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig2_leakage.png"), dpi=300)
plt.close()

# Figure 3: Sensitivity to Sensor Dropout Gap Duration
plt.figure(figsize=(8.5, 5))
for m in df_gap_exp["method"].unique():
    sub = df_gap_exp[df_gap_exp["method"] == m]
    ls = '--' if "Interpolation" in m or "Spline" in m else '-'
    plt.plot(sub["gap_duration_hours"], sub["RMSE_ppm"], marker='o', linestyle=ls, label=m)
plt.xlabel("Contiguous Sensor Dropout Duration [hours]")
plt.ylabel("Reconstruction RMSE [ppm]")
plt.title("Sensitivity to Missing Sensor Block Duration (Numerical vs ML)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig3_dropout_sensitivity.png"), dpi=300)
plt.close()

# Figure 4: Physical Feature Importance Ranking
plt.figure(figsize=(8, 4.2))
top_feat = df_feat_imp.head(8).sort_values("rf_importance", ascending=True)
plt.barh(top_feat["feature"], top_feat["rf_importance"], color=MONASH_BLUE)
plt.xlabel("Gini Feature Importance (Random Forest)")
plt.title("Physical Drivers of Indoor CO2 Concentration")
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "results/figures/fig4_feature_importance.png"), dpi=300)
plt.close()
print("[OK] All 4 Figures saved to results/figures/")

# Repository Zip Compression & Download Hook
zip_path = os.path.join(ROOT_DIR, "MMA3001_Project.zip")
subprocess.run(["zip", "-r", "-q", zip_path, "MMA3001_Project"], cwd=ROOT_DIR, check=True)

print("=" * 80)
print(f"[SUCCESS] Submission archive ready: {zip_path}")
print("=" * 80)

from google.colab import files
files.download(zip_path)


[1/6] Writing modular code files to MMA3001_Project/src/ ...

[2/6] Installing dependencies, running pytest, and generating docs...
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
metadata: {'Python': '3.13.18', 'Platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'Packages': {'pytest': '8.4.2', 'pluggy': '1.7.0'}, 'Plugins': {'html': '4.2.0', 'metadata': '3.1.1', 'langsmith': '0.14.17', 'typeguard': '4.6.0', 'anyio': '4.16.0', 'platformdirs': '4.14.3'}}
rootdir: /content/MMA3001_Project
plugins: html-4.2.0, metadata-3.1.1, langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collecting ... collected 5 items

tests/test_project.py::test_linear_interpolation_exact_on_line PASSED    [ 20%]
tests/test_project.py::test_cubic_spline_exact_on_cubic_polynomial PASSED [ 40%]
tests/test_project.py::test_interpolation_rejects_duplicate_timesta

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>